# Milestone 4 : fiabiliser la solution et analyser les gains

## 1. Suites de tests

Les tests portent sur le moteur du milestone 3, qui est dans le package `backend/` (un module par étape) et que `main.py` enchaîne. Le notebook appelle `main.executer()` : le moteur tourne une fois (planning final, sorties 1 et 2), sans rien afficher ni écrire. Les tests appellent directement les fonctions du backend et les résultats, regroupés sous le nom `m` (`m.consolider_couple`, `m.planning_complet`, `m.sortie2`, ...).

Les tests sont écrits avec **pytest** et exécutés dans le notebook par **ipytest** : chaque cellule `%%ipytest` contient un test (une fonction `test_...` avec des `assert`), et affiche son résultat (`passed` ou `failed` avec le détail). Quand un test a plusieurs cas (`@pytest.mark.parametrize`), chaque cas compte comme un test.

Les mêmes tests existent dans le dossier `tests/` à la racine du projet, pour les lancer hors du notebook (et plus tard dans une CI) : `uv run pytest`. La partie 1.6 les relance tous et donne le bilan.

In [ ]:
import sys

import ipytest

ipytest.autoconfig()
sys.path.insert(0, "..")

import contextlib
import io
import types

import numpy as np
import pandas as pd
import pytest

with contextlib.redirect_stdout(io.StringIO()):   # le moteur tourne une fois, sans rien afficher ni écrire
    import main
    from backend import charge, config, contraintes, couts, donnees, formatage, jours_fixes, lissage, report
    R = main.executer()

# m regroupe les fonctions et données du backend (un seul nom pour les tests) et les résultats du moteur
m = types.SimpleNamespace()
for module in (config, couts, donnees, charge, contraintes, jours_fixes, lissage, report, formatage):
    vars(m).update({nom: valeur for nom, valeur in vars(module).items() if not nom.startswith("_")})
for nom in ("planning_complet", "lignes_complet", "sortie1", "sortie2"):
    setattr(m, nom, getattr(R, nom))

J = pd.Timestamp
P, L = m.planning_complet, m.lignes_complet       # planning final (une ligne par camion) et lignes de besoin livrées


def lignes_test(*lignes):
    # Lignes de besoin d'un couple fictif : (date du besoin, poids en kg, volume en m³)
    return pd.DataFrame([{"need_date": J(d), "weight": w, "volume": v} for d, w, v in lignes])

print(f"Moteur chargé : {len(P)} camions ({int(P['reserve'].sum())} réservés), {len(L)} lignes de besoin, "
      f"du {m.DEBUT:%d/%m} au {m.FIN:%d/%m}")

### 1.1 Tests unitaires des fonctions

Les fonctions clés du moteur sont testées sur des cas construits à la main, dont on connaît le résultat : grille tarifaire, date de départ des camions, consolidation d'un couple, contrôle du lissage. Les dates sont celles du planning (semaine du lundi 16/11/2026, sauf mention contraire).

#### Test 1 : `tranche` : choix de la tranche de la grille

La grille tarifaire est par tranches : on prend la première borne supérieure ou égale à la valeur (on majore). On teste une borne exacte, une valeur juste au-dessus et une valeur au-delà de la dernière borne.

In [ ]:
%%ipytest -vv --no-header

@pytest.mark.parametrize("valeur, tranches, attendu", [
    (30, m.TRANCHES_KM, "30"),        # borne exacte
    (30.1, m.TRANCHES_KM, "50"),      # juste au-dessus : tranche suivante
    (650, m.TRANCHES_KM, "650"),      # dernière borne
    (651, m.TRANCHES_KM, ">650"),     # au-delà de la grille
    (0, m.TRANCHES_KG, "50"),         # poids nul : première tranche
    (4000, m.TRANCHES_KG, "4000"),
    (4000.5, m.TRANCHES_KG, ">4000"),
])
def test_tranche(valeur, tranches, attendu):
    assert m.tranche(valeur, tranches) == attendu

#### Test 2 : `cout_transport` : exemple de la rate sheet

Le prix d'un camion = coût de la grille × (1 + marge de 239,23 %) × surcharge diesel de 1,1943. La rate sheet donne un exemple chiffré : 1 000 kg à 300 km coûtent 335,34 €.

In [ ]:
%%ipytest -vv --no-header

def test_cout_transport_exemple_rate_sheet():
    assert m.cout_transport(1000, 300) == 335.34

#### Test 3 : `cout_transport` : le prix ne baisse jamais quand le poids ou la distance augmente

On calcule le prix sur toute la grille (chaque tranche de poids × chaque tranche de distance, plus les tranches « au-delà »). Un camion plus lourd ou qui va plus loin ne doit jamais coûter moins cher : sinon l'optimisation pourrait gagner de l'argent en se trompant.

In [ ]:
%%ipytest -vv --no-header

def test_cout_transport_croissant():
    poids = m.TRANCHES_KG + [m.POIDS_PAR_DEFAUT]
    distances = m.TRANCHES_KM + [m.DISTANCE_PAR_DEFAUT]
    grille = np.array([[m.cout_transport(w, d) for w in poids] for d in distances])
    assert (np.diff(grille, axis=1) >= 0).all(), "le prix baisse quand le poids augmente"
    assert (np.diff(grille, axis=0) >= 0).all(), "le prix baisse quand la distance augmente"

#### Test 4 : `cout_camion` : valeurs par défaut des données manquantes

Hypothèse des milestones 1 et 2 : un poids inconnu est chiffré dans la tranche > 4 000 kg, une distance inconnue dans la tranche > 650 km. On vérifie que `cout_camion` applique bien ces valeurs par défaut.

In [ ]:
%%ipytest -vv --no-header

def test_cout_camion_valeurs_par_defaut():
    avec_distance = next(s for s in m.distances.index if s not in m.SANS_DISTANCE)
    sans_distance = m.SANS_DISTANCE[0]
    assert m.tranche(m.POIDS_PAR_DEFAUT, m.TRANCHES_KG) == ">4000"
    assert m.tranche(m.DISTANCE_PAR_DEFAUT, m.TRANCHES_KM) == ">650"
    assert m.cout_camion(np.nan, avec_distance) == m.cout_transport(m.POIDS_PAR_DEFAUT, m.distances[avec_distance])
    assert m.cout_camion(1000, sans_distance) == m.cout_transport(1000, m.DISTANCE_PAR_DEFAUT)

#### Test 5 : `date_camion` : jour de départ du camion

Un besoin part le dernier jour fixe du couple avant (ou égal à) sa date. Trois cas : un jour fixe dans la même semaine, un profil à 2 jours, et un besoin du premier jour du planning (lundi 09/11) dont le jour fixe est le jeudi précédent : le camion part dans la semaine ajoutée avant le planning, le jeudi 05/11.

In [ ]:
%%ipytest -vv --no-header

@pytest.mark.parametrize("besoin, profil, attendu", [
    ("2026-11-18", (0,), "2026-11-16"),     # besoin mercredi, livraison le lundi -> lundi 16/11
    ("2026-11-20", (0, 3), "2026-11-19"),   # besoin vendredi, livraison lundi + jeudi -> jeudi 19/11
    ("2026-11-09", (3,), "2026-11-05"),     # besoin du 1er jour, jeudi précédent : semaine ajoutée avant le planning
])
def test_date_camion(besoin, profil, attendu):
    assert m.date_camion(J(besoin), profil) == J(attendu)

#### Test 6 : `date_camion` : règle vérifiée pour tous les profils et tous les jours du planning

Au lieu de quelques exemples, on vérifie la règle pour les 25 profils (1 à 3 jours fixes) et chaque jour de l'horizon, week-end compris : le camion ne part jamais après le besoin ni plus de 7 jours avant, toujours un jour du profil (donc un jour ouvré), et jamais avant le premier jour de livraison possible (lundi 02/11).

In [ ]:
%%ipytest -vv --no-header

@pytest.mark.parametrize("profil", m.PROFILS, ids=m.nom_profil)
def test_date_camion_regle_generale(profil):
    for besoin in pd.date_range(m.DEBUT, m.FIN):
        date = m.date_camion(besoin, profil)
        assert date <= besoin, f"{besoin:%d/%m} : livré en retard"
        assert (besoin - date).days <= 7, f"{besoin:%d/%m} : plus de 7 jours d'avance"
        assert date.day_of_week in profil, f"{besoin:%d/%m} : départ hors des jours fixes"
        assert date >= m.DEBUT_LIVRAISONS, f"{besoin:%d/%m} : départ avant le premier jour de livraison"

#### Test 7 : `prochain_jour_fixe` : jour fixe suivant

Utilisée par le report (bloc 4) : premier jour du profil strictement après une date, éventuellement la semaine suivante.

In [ ]:
%%ipytest -vv --no-header

@pytest.mark.parametrize("date, profil, attendu", [
    ("2026-11-16", (0, 3), "2026-11-19"),   # lundi -> jeudi de la même semaine
    ("2026-11-19", (0, 3), "2026-11-23"),   # jeudi -> lundi suivant
    ("2026-11-16", (0,), "2026-11-23"),     # un seul jour fixe : la semaine suivante
])
def test_prochain_jour_fixe(date, profil, attendu):
    assert m.prochain_jour_fixe(J(date), profil) == J(attendu)

#### Test 8 : `nom_profil` : libellé des jours fixes

Le libellé est celui de la sortie 1 (colonne des jours) : les jours dans l'ordre, séparés par « + ».

In [ ]:
%%ipytest -vv --no-header

@pytest.mark.parametrize("profil, attendu", [((2,), "Mercredi"), ((0, 3), "Lundi + Jeudi")])
def test_nom_profil(profil, attendu):
    assert m.nom_profil(profil) == attendu

#### Test 9 : `profil_utilise` : un couple à plusieurs jours fixes les utilise vraiment (C11)

Un couple au profil « mardi + vendredi » doit avoir au moins une semaine avec un camion le mardi et un camion le vendredi, sinon il n'utilise jamais ses 2 jours. On teste une semaine complète, un jour jamais utilisé, deux jours utilisés mais jamais la même semaine, et un profil d'un seul jour (toujours utilisé).

In [ ]:
%%ipytest -vv --no-header

@pytest.mark.parametrize("dates, profil, attendu", [
    (["2026-11-17", "2026-11-20"], (1, 4), True),                  # mardi + vendredi la même semaine
    (["2026-11-17", "2026-11-24"], (1, 4), False),                 # le vendredi n'est jamais utilisé
    (["2026-11-17", "2026-11-27"], (1, 4), False),                 # mardi et vendredi, mais jamais la même semaine
    (["2026-11-17", "2026-11-18", "2026-11-19"], (1, 2, 3), True), # 3 jours la même semaine
    (["2026-11-17", "2026-11-18"], (1, 2, 3), False),              # 3e jour jamais utilisé
    (["2026-11-18"], (2,), True),                                  # un seul jour : toujours utilisé
])
def test_profil_utilise(dates, profil, attendu):
    assert m.profil_utilise([J(d) for d in dates], profil) is attendu

#### Test 10 : `consolider_couple` : les besoins d'une semaine dans un seul camion

Trois petits besoins du lundi au vendredi (1 t au total), profil « lundi » : ils doivent partir ensemble, dans un seul camion, le lundi.

In [ ]:
%%ipytest -vv --no-header

def test_consolider_couple_regroupe_la_semaine():
    camions = m.consolider_couple(lignes_test(("2026-11-16", 500, 2), ("2026-11-17", 300, 1), ("2026-11-20", 200, 1)), (0,))
    assert len(camions) == 1
    assert camions["date"].iloc[0] == J("2026-11-16")
    assert camions["poids"].iloc[0] == 1000

#### Test 11 : `consolider_couple` : capacité en poids et en volume

Quand la charge dépasse 24,5 t ou 110,4 m³, un deuxième camion est ajouté (SF-07) : 3 besoins de 10 t (30 t) ou 2 besoins de 60 m³ (120 m³) demandent 2 camions, chacun sous la limite.

In [ ]:
%%ipytest -vv --no-header

@pytest.mark.parametrize("cas, lignes", [
    ("poids : 3 x 10 t", [("2026-11-16", 10000, 1), ("2026-11-17", 10000, 1), ("2026-11-18", 10000, 1)]),
    ("volume : 2 x 60 m3", [("2026-11-16", 100, 60), ("2026-11-17", 100, 60)]),
])
def test_consolider_couple_capacite(cas, lignes):
    camions = m.consolider_couple(lignes_test(*lignes), (0,))
    assert len(camions) == 2
    assert (camions["poids"] <= m.POIDS_MAX).all() and (camions["volume"] <= m.VOLUME_MAX).all()

#### Test 12 : `consolider_couple` : fenêtre de 7 jours d'anticipation

Un besoin 7 jours après le camion du lundi part encore dans ce camion (limite autorisée, SF-06). À 8 jours, il part dans un camion suivant.

In [ ]:
%%ipytest -vv --no-header

@pytest.mark.parametrize("jours_apres, nb_camions", [(7, 1), (8, 2)])
def test_consolider_couple_fenetre_7_jours(jours_apres, nb_camions):
    second = J("2026-11-16") + pd.Timedelta(days=jours_apres)
    camions = m.consolider_couple(lignes_test(("2026-11-16", 100, 1), (second, 100, 1)), (0,))
    assert len(camions) == nb_camions

#### Test 13 : `consolider_couple` : aucune ligne perdue ni dupliquée

40 besoins tirés au hasard (graine fixe) sur 4 semaines, profil « mardi + jeudi ». Chaque ligne doit être dans exactement un camion, les poids et volumes conservés, chaque camion sous les limites, et chaque ligne livrée entre 0 et 7 jours avant son besoin.

In [ ]:
%%ipytest -vv --no-header

def test_consolider_couple_conserve_les_lignes():
    rng = np.random.default_rng(0)
    jours_ouvres = pd.date_range("2026-11-09", "2026-12-04", freq="B")
    lignes = lignes_test(*zip(rng.choice(jours_ouvres, 40), rng.uniform(100, 9000, 40), rng.uniform(1, 40, 40)))
    lignes = lignes.sort_values(["need_date", "weight"], ascending=[True, False])
    camions = m.consolider_couple(lignes, (1, 3))
    contenu = [l for c in camions["lignes"] for l in c]
    assert sorted(contenu) == sorted(lignes.index), "ligne perdue ou dupliquée"
    assert np.isclose(camions["poids"].sum(), lignes["weight"].sum())
    assert np.isclose(camions["volume"].sum(), lignes["volume"].sum())
    assert (camions["poids"] <= m.POIDS_MAX).all() and (camions["volume"] <= m.VOLUME_MAX).all()
    for c in camions.itertuples():
        besoins = lignes.loc[c.lignes, "need_date"]
        assert (besoins >= c.date).all() and (besoins <= c.date + m.FENETRE).all()

#### Test 14 : `ecart` : écart d'un jour à la moyenne de la semaine

Mesure utilisée pour le lissage : écart du jour le plus éloigné de la moyenne, en part de la moyenne. Semaine régulière → 0 ; 3-2-2-2-1 (moyenne 2, écart 1) → 50 % ; tout le lundi → 400 % ; semaine vide → 0.

In [ ]:
%%ipytest -vv --no-header

@pytest.mark.parametrize("semaine, attendu", [
    ([2, 2, 2, 2, 2], 0.0),
    ([3, 2, 2, 2, 1], 0.5),
    ([10, 0, 0, 0, 0], 4.0),
    ([0, 0, 0, 0, 0], 0.0),
])
def test_ecart(semaine, attendu):
    assert m.ecart(semaine) == pytest.approx(attendu)

#### Test 15 : `hors_tolerance` : contrôle du lissage d'une semaine

Une semaine est hors tolérance si un jour s'écarte de la moyenne de plus de 35 % et d'au moins 1 camion. Les semaines de moins de 5 camions ne sont pas contrôlées (une usine ne peut pas faire mieux).

In [ ]:
%%ipytest -vv --no-header

@pytest.mark.parametrize("semaine, attendu", [
    ([2, 2, 2, 2, 2], False),    # parfaitement régulière
    ([3, 2, 2, 2, 1], False),    # écart d'un seul camion : toléré
    ([10, 0, 0, 0, 0], True),    # tout le même jour
    ([6, 1, 1, 1, 1], True),     # pic le lundi
    ([4, 0, 0, 0, 0], False),    # moins de 5 camions : pas contrôlée
])
def test_hors_tolerance(semaine, attendu):
    assert m.hors_tolerance(np.array([semaine], dtype=float)).tolist() == [attendu]

#### Test 16 : `dispersion` : critère minimisé par le lissage

Somme des carrés des écarts à la moyenne de chaque semaine contrôlée : nulle quand chaque semaine est régulière, positive dès qu'un jour est chargé. La semaine ajoutée avant le planning et la dernière semaine (incomplètes) ne sont pas contrôlées, donc ne comptent pas.

In [ ]:
%%ipytest -vv --no-header

def test_dispersion():
    reguliere = np.full((m.NB_SEMAINES, 5), 2.0)
    assert m.dispersion(reguliere) == 0
    pic = reguliere.copy()
    pic[m.CONTROLE[0]] = [10, 0, 0, 0, 0]
    assert m.dispersion(pic) > 0
    for semaine in (0, m.NB_SEMAINES - 1):                  # semaine ajoutée et dernière semaine : non contrôlées
        non_controlee = reguliere.copy()
        non_controlee[semaine] = [10, 0, 0, 0, 0]
        assert m.dispersion(non_controlee) == 0

### 1.2 Tests du respect des contraintes

Les contraintes ont été définies au milestone 3 (partie 1). Chacune a sa fonction de vérification, qui retourne les cas qui la violent (table vide = contrainte respectée) :

| # | Contrainte | Type | Fonction |
|---|---|---|---|
| C1 | Un camion ne dépasse pas 24,5 t | dure | `c1_poids` |
| C2 | Un camion ne dépasse pas 110,4 m³ | dure | `c2_volume` |
| C3 | Chaque ligne est livrée, une seule fois, entière | dure | `c3_exhaustivite` |
| C4 | Un camion ne transporte qu'un seul couple | dure | `c4_un_couple` |
| C5 | Jamais de retard | dure | `c5_retard` |
| C6 | Au plus 7 jours d'avance | dure | `c6_avance` |
| C7 | Livraison un jour ouvré, dans l'horizon du planning | dure | `c7_jours_ouvres` |
| C8 | Jours fixes : 1 à 3 jours par couple, le camion part un de ces jours | dure | `c8_jours_fixes` |
| C9 | Au plus 1 camion par couple et par jour, sauf cas impossible (besoin d'un jour > 24,5 t, ou camion réservé d'une ligne sans poids) | dure | `c9_un_camion_par_jour` |
| C10 | Lissage : aucun jour ne s'écarte de plus de 35 % de la moyenne de la semaine | souple | `c10_lissage` |
| C11 | Profil utilisé : un couple à 2 ou 3 jours fixes livre au moins une semaine sur tous ses jours fixes | dure | `c11_profil_utilise` |

On teste deux choses : le planning final respecte les contraintes, et les fonctions de vérification détectent vraiment une erreur. Une fonction qui renverrait toujours une table vide passerait le premier test sans rien vérifier : pour chaque contrainte, on introduit donc volontairement une erreur dans une copie du planning, et on vérifie qu'elle est signalée.

La cellule ci-dessous prépare ces erreurs (une par contrainte).

In [ ]:
FONCTIONS = {code_c: fonction for code_c, _, _, fonction in m.CONTRAINTES}
DURES = [code_c for code_c, _, type_c, _ in m.CONTRAINTES if type_c == "dure"]


def violations(code_c, planning, lignes):
    # Violations d'une contrainte (pour C9 : seulement les cas évitables, sans cause justifiée)
    args = (planning, lignes, m.IDS_ATTENDUS) if code_c == "C3" else (planning, lignes)
    v = FONCTIONS[code_c](*args)
    return v[~v["justifie"]] if code_c == "C9" else v


# ---------- Erreurs introduites volontairement, une par contrainte ----------
opti = P[~P["reserve"]]
i0 = opti.index[0]                                                          # un camion optimisé quelconque
multi = L.groupby("camion_id").size().loc[lambda n: n > 1].index[0]         # un camion qui transporte plusieurs lignes
couples_c9 = set(map(tuple, m.c9_un_camion_par_jour(P, L)[["plant", "supplier_id"]].values))
i9 = next(i for i, r in opti.iterrows() if (r["plant"], r["supplier_id"]) not in couples_c9)
i8 = opti.index[0]
un_jour = opti[opti["profil"].map(len) == 1].iloc[0]                       # un couple à 1 jour fixe (pour C11)


def avec_erreur(code_c):
    # Copie du planning et des lignes avec une erreur qui viole la contrainte
    p, l = P.copy(), L.copy()
    if code_c == "C1":
        p.loc[i0, "poids"] = m.POIDS_MAX + 1                                    # camion de 24,5 t + 1 kg
    elif code_c == "C2":
        p.loc[i0, "volume"] = m.VOLUME_MAX + 1                                  # camion de 110,4 m³ + 1 m³
    elif code_c == "C3":
        l = l.drop(index=l.index[0])                                            # ligne oubliée
    elif code_c == "C4":
        l.loc[l.index[l["camion_id"] == multi][0], "supplier_id"] = "SUP_99999"  # ligne d'un autre fournisseur
    elif code_c == "C5":
        l.loc[l.index[0], "date_livraison"] = l.loc[l.index[0], "need_date"] + pd.Timedelta(days=1)   # 1 jour de retard
    elif code_c == "C6":
        l.loc[l.index[0], "date_livraison"] = l.loc[l.index[0], "need_date"] - pd.Timedelta(days=8)   # 8 jours d'avance
    elif code_c == "C7":
        p.loc[i0, "date"] = J("2026-11-21")                                     # camion un samedi
    elif code_c == "C8":
        date, profil = p.loc[i8, "date"], p.loc[i8, "profil"]                   # camion hors de ses jours fixes
        p.loc[i8, "date"] = date + pd.Timedelta(days=next(j for j in range(5) if j not in profil) - date.day_of_week)
    elif code_c == "C9":
        p = pd.concat([p, p.loc[[i9]]], ignore_index=True)                      # 2 camions le même jour sans raison
    elif code_c == "C11":                                                       # profil de 2 jours dont un jamais utilisé
        autre = next(j for j in range(5) if j not in un_jour["profil"])
        meme_couple = (p["plant"] == un_jour["plant"]) & (p["supplier_id"] == un_jour["supplier_id"]) & ~p["reserve"]
        p["profil"] = [tuple(sorted({*prof, autre})) if cible else prof for prof, cible in zip(p["profil"], meme_couple)]
    elif code_c == "C10":
        renfort = p[p["plant"] == m.USINES[0]].head(1).assign(date=J("2026-11-16"))
        p = pd.concat([p] + [renfort] * 15, ignore_index=True)                  # 15 camions de plus un lundi
    return p, l

#### Test 17 : Le planning final respecte chaque contrainte dure (C1 à C9 et C11)

On rejoue les fonctions de vérification du moteur sur le planning produit par `main.executer()` : aucune violation, pour chacune des 10 contraintes dures (un cas de test par contrainte). Pour C9, les 2 camions d'un même jour dus à un cas impossible (besoin de plus de 24,5 t, ou camion réservé d'une ligne sans poids) ne sont pas des violations.

In [ ]:
%%ipytest -vv --no-header

@pytest.mark.parametrize("contrainte", DURES)
def test_contrainte_dure_respectee(contrainte):
    v = violations(contrainte, P, L)
    assert v.empty, f"{contrainte} : {len(v)} violation(s)\n{v.head()}"

#### Test 18 : C10 lissage : au plus 2 semaines hors tolérance

Le lissage est une contrainte souple : avec le moteur actuel, 2 semaines sur 20 restent hors tolérance. Le test empêche une régression (plus de semaines hors tolérance qu'aujourd'hui).

In [ ]:
%%ipytest -vv --no-header

def test_lissage_c10():
    hors = violations("C10", P, L)
    assert len(hors) <= 2, f"{len(hors)} semaines hors tolérance\n{hors}"

#### Test 19 : C11 : chaque couple à plusieurs jours fixes livre au moins une semaine sur tous ses jours

Contrôle indépendant de la fonction `c11_profil_utilise` : on recalcule, couple par couple, les jours livrés chaque semaine à partir du planning, et on cherche une semaine qui contient tous les jours fixes du profil. Un profil à 2 ou 3 jours qu'on n'utilise jamais en entier serait un faux profil (en réalité, un profil moins cher à moins de jours).

In [ ]:
%%ipytest -vv --no-header

def test_c11_profil_utilise_une_semaine_complete():
    optimises = P[~P["reserve"]]
    assert (optimises["profil"].map(len) >= 2).any(), "aucun couple à plusieurs jours fixes : test sans objet"
    for (plant, supplier), camions in optimises.groupby(["plant", "supplier_id"]):
        profil = set(camions["profil"].iloc[0])
        jours_par_semaine = camions.groupby(camions["date"].dt.isocalendar().week)["date"].apply(
            lambda d: set(d.dt.day_of_week))
        assert any(profil <= jours for jours in jours_par_semaine), (plant, supplier, sorted(profil))

#### Test 20 : Chaque fonction de vérification détecte l'erreur qu'on lui introduit

Pour chaque contrainte, l'erreur préparée plus haut (camion de 24,5 t + 1 kg, ligne en retard, camion un samedi, profil à 2 jours dont un jamais utilisé, ...) doit faire apparaître au moins une violation de plus que sur le planning correct.

In [ ]:
%%ipytest -vv --no-header

@pytest.mark.parametrize("contrainte", list(FONCTIONS))
def test_verification_detecte_erreur(contrainte):
    assert len(violations(contrainte, *avec_erreur(contrainte))) > len(violations(contrainte, P, L))

#### Test 21 : Cohérence du planning : contenu, poids et coût des camions

`valider_resultats` (milestone 3) vérifie que chaque camion contient ce qu'il annonce (poids, volume), que le poids total livré est celui des besoins, qu'il y a un camion réservé par ligne sans poids, et que le coût de chaque camion suit la grille tarifaire.

In [ ]:
%%ipytest -vv --no-header

def test_planning_coherent():
    controles = m.valider_resultats(P, L)
    echecs = controles[controles["Résultat"] != "OK"].index.tolist()
    assert not echecs, echecs

#### Test 22 : C9 : 2 camions le même jour seulement dans un cas impossible

Un couple ne reçoit 2 camions le même jour que dans deux cas : son besoin d'un seul jour dépasse 24,5 t (il ne tient pas dans un camion), ou une de ses lignes sans poids a son camion réservé ce jour-là (règle absolue : elle ne peut être ni regroupée ni déplacée). On recalcule ces cas à partir des lignes livrées.

In [ ]:
%%ipytest -vv --no-header

def test_c9_doublons_justifies():
    doublons = m.c9_un_camion_par_jour(P, L)
    besoin_jour = L.dropna(subset=["weight"]).groupby(["plant", "supplier_id", "need_date"])["weight"].sum()
    couples_lourds = set(besoin_jour[besoin_jour > m.POIDS_MAX].index.droplevel("need_date"))
    optimises = P[~P["reserve"]].groupby(["plant", "supplier_id", "date"]).size()
    for r in doublons.itertuples():
        nb_optimises = optimises.get((r.plant, r.supplier_id, r.date), 0)
        assert nb_optimises <= 1 or (r.plant, r.supplier_id) in couples_lourds, (r.plant, r.supplier_id, r.date)

### 1.3 Cas limites et hypothèses de données manquantes

Les hypothèses prises aux milestones 1 à 3 pour les données incomplètes et les cas particuliers sont vérifiées sur le planning final :
- ligne de plus de 24,5 t : exclue du moteur, en attendant la réponse du client ;
- poids manquant : la ligne ne peut pas être consolidée, elle a son propre camion réservé, le jour du besoin, chiffré au tarif > 4 000 kg ;
- distance manquante : le camion est chiffré au tarif > 650 km, et la distance est vide (« inconnue ») dans la sortie 1 ;
- début du planning : les premiers besoins partent le dernier jour fixe de leur couple, dans la semaine ajoutée avant le planning (à partir du lundi 02/11) ;
- quantité nulle (SF-02) et couple à une seule ligne.

In [ ]:
q = m.quarantaine

#### Test 23 : Ligne de plus de 24,5 t : exclue du moteur

La seule ligne de la quarantaine de plus de 24,5 t ne peut pas tenir dans un camion : elle ne doit apparaître dans aucun camion.

In [ ]:
%%ipytest -vv --no-header

def test_ligne_trop_lourde_exclue():
    lourdes = q[q["weight"] > m.POIDS_MAX]
    assert len(lourdes) == 1, f"{len(lourdes)} ligne(s) de plus de 24,5 t"
    cles = ["need_date", "supplier_id", "material", "plant", "unloading_point"]
    assert lourdes[cles].merge(L[cles]).empty, "la ligne lourde est livrée"

#### Test 24 : Poids manquant : un camion réservé par ligne

Chaque ligne sans poids a son propre camion réservé, livré le jour du besoin et chiffré dans la tranche > 4 000 kg.

In [ ]:
%%ipytest -vv --no-header

def test_poids_manquant_camion_reserve():
    reserves, lignes_r = P[P["reserve"]], L[L["reserve"]]
    assert len(reserves) == q["weight"].isna().sum() == len(lignes_r)
    assert (lignes_r.groupby("camion_id").size() == 1).all()
    assert (lignes_r["date_livraison"] == lignes_r["need_date"]).all()
    attendu = [m.cout_transport(m.POIDS_PAR_DEFAUT, m.distances[s]) for s in reserves["supplier_id"]]
    assert np.allclose(reserves["cout"], attendu)

#### Test 25 : Distance manquante : tarif > 650 km, distance vide dans la sortie 1

Les fournisseurs sans distance sont chiffrés au tarif le plus élevé, pour ne pas surestimer l'économie. Dans la sortie 1, leur distance reste vide (et non 651 km, valeur technique).

In [ ]:
%%ipytest -vv --no-header

def test_distance_manquante():
    camions = P[P["supplier_id"].isin(m.SANS_DISTANCE)]
    assert len(camions) > 0, "aucun fournisseur sans distance dans le planning"
    poids = camions["poids"].fillna(m.POIDS_PAR_DEFAUT)
    assert np.allclose(camions["cout"], [m.cout_transport(w, m.DISTANCE_PAR_DEFAUT) for w in poids])
    assert m.sortie1.loc[m.sortie1["Fournisseur"].isin(m.SANS_DISTANCE), "Distance"].isna().all()

#### Test 26 : Début du planning : semaine ajoutée avant le planning

Les premiers besoins (du lundi 09/11) ont leur jour fixe avant le début du planning : leurs camions partent dans la semaine ajoutée, à partir du lundi 02/11. On vérifie que ces camions existent, qu'aucun ne part avant le 02/11, et qu'ils ne transportent que des besoins du planning.

In [ ]:
%%ipytest -vv --no-header

def test_semaine_ajoutee_avant_le_planning():
    avant = P[P["date"] < m.DEBUT]
    assert len(avant) > 0
    assert (avant["date"] >= m.DEBUT_LIVRAISONS).all(), "camion avant le premier jour de livraison"
    assert not avant["reserve"].any()
    premiere = L[L["camion_id"].isin(avant["camion_id"])].groupby("camion_id")["need_date"].min()
    assert (premiere >= m.DEBUT).all(), "besoin hors planning dans un camion de la semaine ajoutée"

#### Test 27 : SF-02 : aucune ligne de quantité nulle

Le planning brut (bronze) contient des lignes de quantité 0 : elles ont été retirées au milestone 1 et ne doivent être ni livrées ni présentes dans la sortie 2.

In [ ]:
%%ipytest -vv --no-header

def test_quantite_nulle_exclue():
    brut = pd.read_csv(m.DATA / "bronze" / "released_schedule.csv")
    assert (brut["schedule_quantity"] <= 0).any(), "le cas n'existe pas dans les données brutes"
    assert (L["schedule_quantity"] > 0).all()
    assert (m.sortie2["Quantité"] > 0).all()

#### Test 28 : Couple à une seule ligne : un seul camion

Cas minimal de la consolidation : un couple qui n'a qu'un besoin sur toute la période reçoit exactement un camion, qui contient cette ligne.

In [ ]:
%%ipytest -vv --no-header

def test_couple_a_une_seule_ligne():
    assert len(m.consolider_couple(lignes_test(("2026-11-18", 500, 2)), (0,))) == 1
    nb_lignes = m.besoins.groupby(["plant", "supplier_id"]).size()
    seuls = nb_lignes[nb_lignes == 1].index
    assert len(seuls) > 0
    for plant, supplier in seuls:
        camions = P[(P["plant"] == plant) & (P["supplier_id"] == supplier) & ~P["reserve"]]
        assert len(camions) == 1 and camions["nb_lignes"].iloc[0] == 1, (plant, supplier)

### 1.4 Exhaustivité des matériaux livrés et anticipation des besoins

Le moteur réorganise les livraisons (il regroupe des besoins de plusieurs jours dans un même camion), mais il ne doit rien perdre, rien inventer et ne jamais mettre une usine en rupture :
- exhaustivité : chaque ligne de besoin du planning d'entrée (silver + quarantaine) se retrouve dans un camion, une seule fois ;
- matériaux : pour chaque material, la quantité demandée par l'usine est égale à la quantité chargée dans les camions (sortie 2) ;
- anticipation : chaque ligne arrive au plus tard le jour de son besoin (pas de rupture), et au plus 7 jours avant (pas plus d'une semaine de stock).

La cellule ci-dessous prépare la comparaison entre le planning d'entrée et la sortie 2.

In [ ]:
CLES = ["plant", "supplier_id", "material", "unloading_point"]
entree = pd.concat([m.besoins, m.lignes_reserves])                          # planning d'entrée : besoins à livrer
demande = entree.groupby(CLES)["schedule_quantity"].sum()                   # quantité demandée par material
livre = (m.sortie2.rename(columns={"usine": "plant", "fournisseur": "supplier_id", "Material": "material", "Quai": "unloading_point"})
         .groupby(CLES)["Quantité"].sum())                                  # quantité chargée dans les camions (sortie 2)

#### Test 29 : Aucune ligne oubliée ni livrée deux fois

Les lignes livrées sont exactement celles du planning d'entrée, chacune une fois et dans un camion. La sortie 2 compte toutes ces lignes et tous les camions.

In [ ]:
%%ipytest -vv --no-header

def test_aucune_ligne_oubliee_ni_doublee():
    assert L["id_ligne"].is_unique
    assert set(L["id_ligne"]) == m.IDS_ATTENDUS
    assert L["camion_id"].notna().all()
    assert m.sortie2["nb_lignes"].sum() == len(m.IDS_ATTENDUS)
    assert set(m.sortie2["camion_id"]) == set(P["camion_id"])

#### Test 30 : Chaque material est livré en quantité exacte

Pour chaque combinaison material × quai × couple, la quantité demandée (planning d'entrée) est égale à la quantité chargée dans les camions (sortie 2) : le regroupement change les jours, pas les quantités.

In [ ]:
%%ipytest -vv --no-header

def test_quantite_exacte_par_material():
    assert set(demande.index) == set(livre.index), "material manquant ou inventé"
    ecarts = demande.sub(livre, fill_value=0).abs()
    assert (ecarts < 1e-6).all(), f"{(ecarts >= 1e-6).sum()} materials avec un écart"

#### Test 31 : Le poids total livré est égal au poids demandé

Contrôle global sur les lignes au poids connu : la somme des poids livrés (lignes et sortie 2) est égale à celle des besoins. La sortie 2 arrondit les poids à 0,1 kg, d'où la petite tolérance.

In [ ]:
%%ipytest -vv --no-header

def test_poids_total_conserve():
    poids_demande = m.besoins["weight"].sum()
    assert np.isclose(L["weight"].sum(), poids_demande)
    assert abs(m.sortie2["Poids (kg)"].sum() - poids_demande) <= 0.05 * len(m.sortie2)

#### Test 32 : Anticipation : chaque ligne livrée entre 0 et 7 jours avant son besoin

Aucune usine en rupture (jamais de retard) et pas plus d'une semaine de stock (au plus 7 jours d'avance), pour chacune des lignes livrées.

In [ ]:
%%ipytest -vv --no-header

def test_anticipation_des_besoins():
    avance = (L["need_date"] - L["date_livraison"]).dt.days
    assert (avance == L["avance_jours"]).all()
    assert avance.between(0, 7).all(), avance.describe().to_string()

Le graphique montre avec combien de jours d'avance chaque ligne est livrée, du « jour même » à « 7 jours avant », la limite autorisée.

In [ ]:
import matplotlib.pyplot as plt

avance = L["avance_jours"].value_counts().sort_index().reindex(range(8), fill_value=0)
part = avance / avance.sum()
fig, ax = plt.subplots(figsize=(9, 4.5))
barres = ax.bar(avance.index, part * 100, color="royalblue", edgecolor="black")
ax.bar_label(barres, labels=[f"{m.pourcent(p)}\n({n} lignes)" for p, n in zip(part, avance)], fontsize=8, padding=2)
ax.set_xticks(range(8), ["le jour\nmême"] + [f"{j} j avant" for j in range(1, 8)])
ax.set_xlabel("Date de livraison par rapport à la date du besoin")
ax.set_ylabel("Part des lignes de besoin (%)")
ax.set_title(f"Avance des livraisons sur le besoin ({len(L)} lignes) : jamais de retard, jamais plus de 7 jours")
ax.margins(y=0.2)
ax.grid(axis="y", alpha=0.3)
plt.show()

#### Test 33 : Chaque camion part avant tous les besoins qu'il transporte

Vu côté camion : une seule date de livraison par camion, au plus tard le jour du premier besoin qu'il transporte, et au plus 7 jours avant le dernier.

In [ ]:
%%ipytest -vv --no-header

def test_camion_dans_la_fenetre_de_ses_besoins():
    par_camion = L.groupby("camion_id").agg(premier=("need_date", "min"), dernier=("need_date", "max"),
                                            date=("date_livraison", "first"), nb_dates=("date_livraison", "nunique"))
    assert (par_camion["nb_dates"] == 1).all(), "un camion avec plusieurs dates de livraison"
    assert (par_camion["date"] <= par_camion["premier"]).all()
    assert ((par_camion["dernier"] - par_camion["date"]).dt.days <= 7).all()
    assert (par_camion["date"] == P.set_index("camion_id").loc[par_camion.index, "date"]).all()

### 1.5 Format des livrables (`data/gold`)

Les deux livrables sont relus depuis les fichiers CSV de `data/gold`, comme le ferait l'utilisateur :
- `sortie1_profils.csv` : le profil de chaque couple (fournisseur, usine, distance, livraisons par semaine actuelles, fréquence cible, jours, camions par semaine, remplissage poids) ;
- `sortie2_camions.csv` : le contenu de chaque camion (material, quai, besoins regroupés, quantité, poids), une ligne par material d'un camion.

Les tests vérifient le contenu, pas la mise en forme : le CSV garde des valeurs brutes (121.0 et non « 121 km », 0.476 et non « 47,6 % »), la mise en forme est faite à l'affichage. Les besoins regroupés d'un material sont une liste de dates dans une seule cellule (par exemple « 2026-11-16, 2026-11-17 »).

Les noms de colonnes attendus sont définis ci-dessous : s'ils changent, il suffit de les modifier ici.

In [ ]:
SORTIE1 = m.SORTIE / "sortie1_profils.csv"
SORTIE2 = m.SORTIE / "sortie2_camions.csv"
COLONNES_SORTIE1 = ["Fournisseur", "Usine", "Distance", "Livraisons / sem. actuelles", "Fréquence cible", "Jours",
                    "Camions / sem.", "Remplissage poids"]
COLONNES_SORTIE2 = ["camion_id", "usine", "fournisseur", "date_livraison", "jour", "Material", "Quai", "Besoins regroupés",
                    "nb_lignes", "Quantité", "Poids (kg)", "reserve"]


def lire_sortie1():
    assert SORTIE1.exists(), f"{SORTIE1} absent : lancer python main.py"
    return pd.read_csv(SORTIE1)


def lire_sortie2():
    assert SORTIE2.exists(), f"{SORTIE2} absent : lancer python main.py"
    return pd.read_csv(SORTIE2, parse_dates=["date_livraison"], date_format="%Y-%m-%d")

#### Test 34 : Sortie 1 : colonnes et une ligne par couple

Le fichier existe, a les colonnes attendues dans l'ordre, et contient exactement une ligne (un profil) par couple fournisseur × usine du planning.

In [ ]:
%%ipytest -vv --no-header

def test_sortie1_structure():
    s1 = lire_sortie1()
    assert list(s1.columns) == COLONNES_SORTIE1
    assert not s1.duplicated(["Fournisseur", "Usine"]).any(), "couple en double"
    assert len(s1) == m.besoins.groupby(["plant", "supplier_id"]).ngroups

#### Test 35 : Sortie 1 : valeurs valides

Identifiants au bon format, distance positive (vide seulement pour un fournisseur sans distance), fréquence de 1 à 3, jours fixes valides, distincts, dans l'ordre de la semaine et en nombre égal à la fréquence, camions par semaine positifs, remplissage entre 0 et 100 %.

In [ ]:
%%ipytest -vv --no-header

def test_sortie1_valeurs():
    s1 = lire_sortie1()
    assert s1["Fournisseur"].str.fullmatch(r"SUP_\d{5}").all()
    assert s1["Usine"].str.fullmatch(r"PLT_\d{3}").all()
    sans = s1["Fournisseur"].isin(m.SANS_DISTANCE)
    assert s1.loc[sans, "Distance"].isna().all() and (s1.loc[~sans, "Distance"] > 0).all()
    assert (s1["Livraisons / sem. actuelles"] > 0).all()
    assert s1["Fréquence cible"].isin(range(1, m.JOURS_FIXES_MAX + 1)).all()
    jours = s1["Jours"].str.split(" + ", regex=False)
    assert jours.map(lambda j: set(j) <= set(m.JOURS)).all(), "jour inconnu"
    assert jours.map(lambda j: [m.JOURS.index(x) for x in j] == sorted({m.JOURS.index(x) for x in j})).all()
    assert (jours.map(len) == s1["Fréquence cible"]).all(), "nombre de jours différent de la fréquence"
    assert (s1["Camions / sem."] > 0).all()
    assert s1["Remplissage poids"].between(0, 1, inclusive="right").all()

#### Test 36 : Sortie 2 : colonnes, types et valeurs

Colonnes attendues, numéro de camion et nombre de lignes entiers, dates au format AAAA-MM-JJ et jour de la semaine cohérent, identifiants au bon format, quantité positive, poids renseigné sauf pour un camion réservé, et une seule ligne par material × quai dans un camion.

In [ ]:
%%ipytest -vv --no-header

def test_sortie2_structure():
    s2 = lire_sortie2()
    assert list(s2.columns) == COLONNES_SORTIE2
    assert pd.api.types.is_integer_dtype(s2["camion_id"]) and pd.api.types.is_integer_dtype(s2["nb_lignes"])
    assert pd.api.types.is_datetime64_any_dtype(s2["date_livraison"]) and s2["date_livraison"].notna().all()
    assert (s2["jour"] == s2["date_livraison"].dt.day_of_week.map(dict(enumerate(m.JOURS)))).all()
    assert s2["usine"].str.fullmatch(r"PLT_\d{3}").all() and s2["fournisseur"].str.fullmatch(r"SUP_\d{5}").all()
    assert s2["Material"].str.fullmatch(r"MAT_\d{6}").all() and s2["Quai"].str.fullmatch(r"ULP_\d{4}").all()
    assert (s2["Quantité"] > 0).all() and (s2["nb_lignes"] >= 1).all()
    assert s2["reserve"].dtype == bool
    assert s2.loc[s2["reserve"], "Poids (kg)"].isna().all()
    assert (s2.loc[~s2["reserve"], "Poids (kg)"] >= 0).all()
    assert not s2.duplicated(["camion_id", "Material", "Quai"]).any()

#### Test 37 : Sortie 2 : besoins regroupés dans une seule cellule

La colonne des besoins regroupés contient, dans une seule cellule, la liste des dates de besoin du material livrées par ce camion. Chaque liste doit être lisible (dates valides, distinctes, triées), compter au plus autant de dates que de lignes, et chaque date doit être entre le jour de livraison et 7 jours après.

In [ ]:
%%ipytest -vv --no-header

def test_sortie2_besoins_regroupes():
    s2 = lire_sortie2()
    erreurs = []
    for i, (besoins, livraison, nb) in enumerate(zip(s2["Besoins regroupés"], s2["date_livraison"], s2["nb_lignes"])):
        dates = pd.to_datetime(besoins.split(", "), format="%Y-%m-%d")
        if not (1 <= len(dates) <= nb and dates.is_unique and dates.is_monotonic_increasing):
            erreurs.append((i, besoins, "liste mal formée"))
        elif dates.min() < livraison or (dates.max() - livraison).days > 7:
            erreurs.append((i, besoins, f"hors fenêtre de la livraison du {livraison:%d/%m}"))
    assert not erreurs, erreurs[:5]

#### Test 38 : Les fichiers sont à jour avec le moteur

Les CSV relus doivent être identiques aux sorties calculées par `main.py` : sinon le livrable est périmé (moteur modifié sans relancer l'export).

In [ ]:
%%ipytest -vv --no-header

def test_livrables_a_jour():
    pd.testing.assert_frame_equal(lire_sortie1(), m.sortie1, check_dtype=False)
    pd.testing.assert_frame_equal(lire_sortie2(), m.sortie2, check_dtype=False)

### 1.6 Résultats des tests

On relance toute la suite du dossier `tests/` avec pytest (les mêmes tests que ci-dessus, un fichier par section). Le tableau résume chaque test : **OK** si tous ses cas passent, **NOK** sinon (le détail de l'échec est affiché sous le tableau).

In [ ]:
DESCRIPTIONS = [(1, '1.1', 'test_tranche', 'tranche : choix de la tranche de la grille', 'La grille tarifaire est par tranches : on prend la première borne supérieure ou égale à la valeur (on majore).'), (2, '1.1', 'test_cout_transport_exemple_rate_sheet', 'cout_transport : exemple de la rate sheet', "Le prix d'un camion = coût de la grille × (1 + marge de 239,23 %) × surcharge diesel de 1,1943."), (3, '1.1', 'test_cout_transport_croissant', 'cout_transport : le prix ne baisse jamais quand le poids ou la distance augmente', 'On calcule le prix sur toute la grille (chaque tranche de poids × chaque tranche de distance, plus les tranches « au-delà »).'), (4, '1.1', 'test_cout_camion_valeurs_par_defaut', 'cout_camion : valeurs par défaut des données manquantes', 'Hypothèse des milestones 1 et 2 : un poids inconnu est chiffré dans la tranche > 4 000 kg, une distance inconnue dans la tranche > 650 km.'), (5, '1.1', 'test_date_camion', 'date_camion : jour de départ du camion', 'Un besoin part le dernier jour fixe du couple avant (ou égal à) sa date.'), (6, '1.1', 'test_date_camion_regle_generale', 'date_camion : règle vérifiée pour tous les profils et tous les jours du planning', "Au lieu de quelques exemples, on vérifie la règle pour les 25 profils (1 à 3 jours fixes) et chaque jour de l'horizon, week-end compris : le camion ne part jamais après le besoin ni plus de 7 jours avant, toujours un jour du profil (donc un jour ouvré), et jamais avant le premier jour de livraison possible (lundi 02/11)."), (7, '1.1', 'test_prochain_jour_fixe', 'prochain_jour_fixe : jour fixe suivant', 'Utilisée par le report (bloc 4) : premier jour du profil strictement après une date, éventuellement la semaine suivante.'), (8, '1.1', 'test_nom_profil', 'nom_profil : libellé des jours fixes', "Le libellé est celui de la sortie 1 (colonne des jours) : les jours dans l'ordre, séparés par « + »."), (9, '1.1', 'test_profil_utilise', 'profil_utilise : un couple à plusieurs jours fixes les utilise vraiment (C11)', "Un couple au profil « mardi + vendredi » doit avoir au moins une semaine avec un camion le mardi et un camion le vendredi, sinon il n'utilise jamais ses 2 jours."), (10, '1.1', 'test_consolider_couple_regroupe_la_semaine', "consolider_couple : les besoins d'une semaine dans un seul camion", 'Trois petits besoins du lundi au vendredi (1 t au total), profil « lundi » : ils doivent partir ensemble, dans un seul camion, le lundi.'), (11, '1.1', 'test_consolider_couple_capacite', 'consolider_couple : capacité en poids et en volume', 'Quand la charge dépasse 24,5 t ou 110,4 m³, un deuxième camion est ajouté (SF-07) : 3 besoins de 10 t (30 t) ou 2 besoins de 60 m³ (120 m³) demandent 2 camions, chacun sous la limite.'), (12, '1.1', 'test_consolider_couple_fenetre_7_jours', "consolider_couple : fenêtre de 7 jours d'anticipation", 'Un besoin 7 jours après le camion du lundi part encore dans ce camion (limite autorisée, SF-06).'), (13, '1.1', 'test_consolider_couple_conserve_les_lignes', 'consolider_couple : aucune ligne perdue ni dupliquée', '40 besoins tirés au hasard (graine fixe) sur 4 semaines, profil « mardi + jeudi ».'), (14, '1.1', 'test_ecart', "ecart : écart d'un jour à la moyenne de la semaine", 'Mesure utilisée pour le lissage : écart du jour le plus éloigné de la moyenne, en part de la moyenne.'), (15, '1.1', 'test_hors_tolerance', "hors_tolerance : contrôle du lissage d'une semaine", "Une semaine est hors tolérance si un jour s'écarte de la moyenne de plus de 35 % et d'au moins 1 camion."), (16, '1.1', 'test_dispersion', 'dispersion : critère minimisé par le lissage', "Somme des carrés des écarts à la moyenne de chaque semaine contrôlée : nulle quand chaque semaine est régulière, positive dès qu'un jour est chargé."), (17, '1.2', 'test_contrainte_dure_respectee', 'Le planning final respecte chaque contrainte dure (C1 à C9 et C11)', 'On rejoue les fonctions de vérification du moteur sur le planning produit par main.executer() : aucune violation, pour chacune des 10 contraintes dures (un cas de test par contrainte).'), (18, '1.2', 'test_lissage_c10', 'C10 lissage : au plus 2 semaines hors tolérance', 'Le lissage est une contrainte souple : avec le moteur actuel, 2 semaines sur 20 restent hors tolérance.'), (19, '1.2', 'test_c11_profil_utilise_une_semaine_complete', 'C11 : chaque couple à plusieurs jours fixes livre au moins une semaine sur tous ses jours', 'Contrôle indépendant de la fonction c11_profil_utilise : on recalcule, couple par couple, les jours livrés chaque semaine à partir du planning, et on cherche une semaine qui contient tous les jours fixes du profil.'), (20, '1.2', 'test_verification_detecte_erreur', "Chaque fonction de vérification détecte l'erreur qu'on lui introduit", "Pour chaque contrainte, l'erreur préparée plus haut (camion de 24,5 t + 1 kg, ligne en retard, camion un samedi, profil à 2 jours dont un jamais utilisé, ...) doit faire apparaître au moins une violation de plus que sur le planning correct."), (21, '1.2', 'test_planning_coherent', 'Cohérence du planning : contenu, poids et coût des camions', "valider_resultats (milestone 3) vérifie que chaque camion contient ce qu'il annonce (poids, volume), que le poids total livré est celui des besoins, qu'il y a un camion réservé par ligne sans poids, et que le coût de chaque camion suit la grille tarifaire."), (22, '1.2', 'test_c9_doublons_justifies', 'C9 : 2 camions le même jour seulement dans un cas impossible', "Un couple ne reçoit 2 camions le même jour que dans deux cas : son besoin d'un seul jour dépasse 24,5 t (il ne tient pas dans un camion), ou une de ses lignes sans poids a son camion réservé ce jour-là (règle absolue : elle ne peut être ni regroupée ni déplacée)."), (23, '1.3', 'test_ligne_trop_lourde_exclue', 'Ligne de plus de 24,5 t : exclue du moteur', 'La seule ligne de la quarantaine de plus de 24,5 t ne peut pas tenir dans un camion : elle ne doit apparaître dans aucun camion.'), (24, '1.3', 'test_poids_manquant_camion_reserve', 'Poids manquant : un camion réservé par ligne', 'Chaque ligne sans poids a son propre camion réservé, livré le jour du besoin et chiffré dans la tranche > 4 000 kg.'), (25, '1.3', 'test_distance_manquante', 'Distance manquante : tarif > 650 km, distance vide dans la sortie 1', "Les fournisseurs sans distance sont chiffrés au tarif le plus élevé, pour ne pas surestimer l'économie."), (26, '1.3', 'test_semaine_ajoutee_avant_le_planning', 'Début du planning : semaine ajoutée avant le planning', 'Les premiers besoins (du lundi 09/11) ont leur jour fixe avant le début du planning : leurs camions partent dans la semaine ajoutée, à partir du lundi 02/11.'), (27, '1.3', 'test_quantite_nulle_exclue', 'SF-02 : aucune ligne de quantité nulle', 'Le planning brut (bronze) contient des lignes de quantité 0 : elles ont été retirées au milestone 1 et ne doivent être ni livrées ni présentes dans la sortie 2.'), (28, '1.3', 'test_couple_a_une_seule_ligne', 'Couple à une seule ligne : un seul camion', "Cas minimal de la consolidation : un couple qui n'a qu'un besoin sur toute la période reçoit exactement un camion, qui contient cette ligne."), (29, '1.4', 'test_aucune_ligne_oubliee_ni_doublee', 'Aucune ligne oubliée ni livrée deux fois', "Les lignes livrées sont exactement celles du planning d'entrée, chacune une fois et dans un camion."), (30, '1.4', 'test_quantite_exacte_par_material', 'Chaque material est livré en quantité exacte', "Pour chaque combinaison material × quai × couple, la quantité demandée (planning d'entrée) est égale à la quantité chargée dans les camions (sortie 2) : le regroupement change les jours, pas les quantités."), (31, '1.4', 'test_poids_total_conserve', 'Le poids total livré est égal au poids demandé', 'Contrôle global sur les lignes au poids connu : la somme des poids livrés (lignes et sortie 2) est égale à celle des besoins.'), (32, '1.4', 'test_anticipation_des_besoins', 'Anticipation : chaque ligne livrée entre 0 et 7 jours avant son besoin', "Aucune usine en rupture (jamais de retard) et pas plus d'une semaine de stock (au plus 7 jours d'avance), pour chacune des lignes livrées."), (33, '1.4', 'test_camion_dans_la_fenetre_de_ses_besoins', "Chaque camion part avant tous les besoins qu'il transporte", "Vu côté camion : une seule date de livraison par camion, au plus tard le jour du premier besoin qu'il transporte, et au plus 7 jours avant le dernier."), (34, '1.5', 'test_sortie1_structure', 'Sortie 1 : colonnes et une ligne par couple', "Le fichier existe, a les colonnes attendues dans l'ordre, et contient exactement une ligne (un profil) par couple fournisseur × usine du planning."), (35, '1.5', 'test_sortie1_valeurs', 'Sortie 1 : valeurs valides', "Identifiants au bon format, distance positive (vide seulement pour un fournisseur sans distance), fréquence de 1 à 3, jours fixes valides, distincts, dans l'ordre de la semaine et en nombre égal à la fréquence, camions par semaine positifs, remplissage entre 0 et 100 %."), (36, '1.5', 'test_sortie2_structure', 'Sortie 2 : colonnes, types et valeurs', 'Colonnes attendues, numéro de camion et nombre de lignes entiers, dates au format AAAA-MM-JJ et jour de la semaine cohérent, identifiants au bon format, quantité positive, poids renseigné sauf pour un camion réservé, et une seule ligne par material × quai dans un camion.'), (37, '1.5', 'test_sortie2_besoins_regroupes', 'Sortie 2 : besoins regroupés dans une seule cellule', 'La colonne des besoins regroupés contient, dans une seule cellule, la liste des dates de besoin du material livrées par ce camion.'), (38, '1.5', 'test_livrables_a_jour', 'Les fichiers sont à jour avec le moteur', "Les CSV relus doivent être identiques aux sorties calculées par main.py : sinon le livrable est périmé (moteur modifié sans relancer l'export).")]


class Collecteur:
    # Plugin pytest : garde le résultat de chaque cas de test
    def __init__(self):
        self.cas = []

    def pytest_runtest_logreport(self, report):
        if report.when == "call" or (report.when == "setup" and report.outcome != "passed"):
            self.cas.append((report.nodeid.split("::")[-1].split("[")[0], report.outcome == "passed"))


collecteur = Collecteur()
sortie_pytest = io.StringIO()
with contextlib.redirect_stdout(sortie_pytest):   # le détail de pytest n'est affiché qu'en cas d'échec
    code_retour = pytest.main(["-q", "--no-header", "-p", "no:cacheprovider", "../tests"], plugins=[collecteur])

# ---------- Tableau récapitulatif : un test par ligne, OK si tous ses cas passent ----------
cas = pd.DataFrame(collecteur.cas, columns=["fonction", "ok"]).groupby("fonction")["ok"].agg(["size", "all"])
bilan = pd.DataFrame(DESCRIPTIONS, columns=["N°", "Section", "fonction", "Test", "Ce qu'il vérifie"])
bilan["Cas"] = bilan["fonction"].map(cas["size"]).fillna(0).astype(int)
bilan["Résultat"] = ["OK" if f in cas.index and cas.loc[f, "all"] else "NOK" for f in bilan["fonction"]]
bilan = bilan.drop(columns="fonction")

couleurs = lambda v: ("color: #1b7f3b; font-weight: bold" if v == "OK" else "color: #b00020; font-weight: bold")
display(bilan.style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "normal"})
        .map(couleurs, subset=["Résultat"]))
print(f"{(bilan['Résultat'] == 'OK').sum()} tests OK sur {len(bilan)} ({bilan['Cas'].sum()} cas). "
      + ("Tous les tests passent." if code_retour == 0 else "Des tests échouent : voir les lignes NOK."))
if code_retour != 0:
    print(sortie_pytest.getvalue())

Résultat : les 38 tests passent, soit 106 cas en comptant chaque cas des tests paramétrés.

- **Fonctions** : les fonctions clés du moteur donnent les résultats attendus sur des cas construits à la main, y compris aux bornes. La tranche exacte est retrouvée, 7 jours d'avance sont acceptés et 8 refusés, et un deuxième camion est ajouté dès que la capacité en poids ou en volume est dépassée. Le prix calculé retrouve l'exemple de la rate sheet (335,34 €), et il ne baisse jamais quand le poids ou la distance augmente. La règle du jour de départ est vérifiée pour les 25 profils et chaque jour du planning, et `profil_utilise` (C11) distingue un profil utilisé d'un profil dont un jour n'est jamais livré.
- **Contraintes** : le planning final respecte les 10 contraintes dures (C1 à C9 et C11), et le lissage reste à 2 semaines hors tolérance sur 20. Pour C11, chaque couple à 2 ou 3 jours fixes livre au moins une semaine sur tous ses jours. Chaque fonction de vérification détecte l'erreur qu'on lui introduit, donc les contrôles du milestone 3 sont fiables.
- **Données manquantes** : les hypothèses sont appliquées comme prévu. La ligne de plus de 24,5 t est exclue, chaque ligne sans poids a son camion réservé, et les fournisseurs sans distance sont chiffrés au tarif le plus élevé. Les lignes de quantité nulle ne sont pas livrées.
- **Exhaustivité et anticipation** : rien n'est perdu ni inventé. Chaque ligne est livrée une fois, et la quantité livrée est égale à la quantité demandée pour chaque material, quai et couple. Aucune usine n'est livrée en retard, ni plus de 7 jours avant son besoin.
- **Livrables** : `sortie1_profils.csv` et `sortie2_camions.csv` ont les colonnes attendues et des valeurs valides. Les besoins regroupés tiennent dans une seule cellule, et les fichiers sont à jour avec le moteur. Un fichier modifié à la main, par exemple avec une fréquence de 4, fait échouer les tests.

## 2. Analyse comparative (avant / après)

Cette partie compare la situation actuelle (baseline du milestone 2) au planning consolidé produit par l'algorithme du milestone 3. 
Les deux sont chargés depuis leurs fichiers. 
Les mêmes indicateurs sont calculés des deux côtés, avec la même fonction (`preparer`) : coût avec la grille tarifaire du milestone 2, taux de remplissage, semaine et jour.

Les **134 camions réservés** (lignes sans poids) sont identiques avant et après. Ils sont **exclus des distributions** de remplissage, car on ne connaît pas leur chargement, mais **inclus dans les coûts totaux**.

Dans un premier temps, nous récupérons tous les fichiers nécessaires : la baseline et notre planning optimisé.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#  Fichiers 
BASELINE_PATH = "../data/baseline/baseline_camion.csv"     # avant : une ligne par camion (milestone 2)
SORTIE2_PATH = "../data/gold/sortie2_camions.csv"           # après : une ligne par camion × material × quai (milestone 3)
SILVER_CLEAN_PATH = "../data/silver/released_schedule_cleaned.csv"
QUARANTAINE_PATH = "../data/silver/released_schedule_quarantine.csv"
RATE_SHEET_PATH = "../data/bronze/rate_sheet.csv"
DISTANCES_PATH = "../data/bronze/distances.csv"

# Paramètres (mêmes valeurs qu'aux milestones 2 et 3)
POIDS_MAX = 24500     # kg
VOLUME_MAX = 110.4    # m³
FREIGHT_VALUE_MARGIN = 2.3923   # +239,23 %
DIESEL_SURCHARGE = 1.1943

COULEURS = {"Avant": "#b8c4cc", "Après": "#3f7489"}   # gris = baseline, bleu = planning consolidé

#  Grille tarifaire 
couts = pd.read_csv(RATE_SHEET_PATH, index_col="km_max")
TRANCHES_KM = [int(km) for km in couts.index if not km.startswith(">")]
TRANCHES_KG = [int(kg) for kg in couts.columns if not kg.startswith(">")]
POIDS_PAR_DEFAUT = TRANCHES_KG[-1] + 1      # poids inconnu -> tranche > 4 000 kg
DISTANCE_PAR_DEFAUT = TRANCHES_KM[-1] + 1   # distance inconnue -> tranche > 650 km


def tranche(valeur, tranches):
    # Première tranche >= valeur (on majore), sinon la tranche '>max'
    for borne in tranches:
        if valeur <= borne:
            return str(borne)
    return f">{tranches[-1]}"


def cout_transport(poids, distance):
    # Prix d'un camion en € = coût de la grille × (1 + marge) × surcharge diesel
    cout_base = couts.loc[tranche(distance, TRANCHES_KM), tranche(poids, TRANCHES_KG)]
    return float(round(cout_base * (1 + FREIGHT_VALUE_MARGIN) * DIESEL_SURCHARGE, 2))


# Mise en forme 
def nombre(x, decimales=1):
    return f"{x:,.{decimales}f}".replace(",", " ").replace(".", ",").removesuffix(",0")
def pourcent(x, decimales=1):
    return f"{x:.{decimales}%}".replace(".", ",").replace("%", " %")
def eur(x):
    return f"{x:,.0f} €".replace(",", " ")


#  Avant : baseline du milestone 2 
distances = pd.read_csv(DISTANCES_PATH, encoding="utf-8-sig").set_index("supplier_id")["distance"]
avant = pd.read_csv(BASELINE_PATH, parse_dates=["date"])[["camion_id", "supplier_id", "plant", "date", "poids", "volume", "reserve"]]

#  Après : planning consolidé du milestone 3, reconstruit camion par camion depuis la sortie 2 
sortie2 = pd.read_csv(SORTIE2_PATH, parse_dates=["date_livraison"])
sortie2 = sortie2.rename(columns={"Material": "material", "Quai": "quai", "Besoins regroupés": "besoins",
                                  "Quantité": "quantite", "Poids (kg)": "poids_kg"})   # noms des colonnes du livrable

# La sortie 2 n'a pas le volume : on le reprend dans les lignes de besoin (silver + quarantaine).
# Une ligne de la sortie 2 peut regrouper plusieurs dates de besoin ("2026-11-16, 2026-11-17") : on la découpe par date.
besoins = pd.concat([pd.read_csv(SILVER_CLEAN_PATH, parse_dates=["need_date"]),
                     pd.read_csv(QUARANTAINE_PATH, parse_dates=["need_date"])], ignore_index=True)
CLE = ["need_date", "supplier_id", "plant", "material", "unloading_point"]
volumes = besoins.groupby(CLE)["volume"].sum(min_count=1)

detail = (sortie2.rename(columns={"fournisseur": "supplier_id", "usine": "plant", "quai": "unloading_point"})
                 .assign(need_date=lambda d: d["besoins"].str.split(", "))
                 .explode("need_date"))
detail["need_date"] = pd.to_datetime(detail["need_date"])
detail["volume"] = volumes.reindex(pd.MultiIndex.from_frame(detail[CLE])).values

apres = (detail.groupby("camion_id")
               .agg(supplier_id=("supplier_id", "first"), plant=("plant", "first"), date=("date_livraison", "first"),
                    volume=("volume", lambda v: v.sum(min_count=1)), reserve=("reserve", "first"))
               .reset_index())
apres["poids"] = sortie2.groupby("camion_id")["poids_kg"].sum(min_count=1).loc[apres["camion_id"]].values  # NaN = réservé


#  Indicateurs communs avant/après
def preparer(camions, version):
    c = camions.copy()
    c["version"] = version
    c["distance"] = c["supplier_id"].map(distances).fillna(DISTANCE_PAR_DEFAUT)   # sans distance -> > 650 km
    c["cout"] = [cout_transport(POIDS_PAR_DEFAUT if r else p, d)                   # réservé -> > 4 000 kg
                 for p, d, r in zip(c["poids"], c["distance"], c["reserve"])]
    c["remplissage_poids"] = c["poids"] / POIDS_MAX
    c["remplissage_volume"] = c["volume"] / VOLUME_MAX
    c["semaine"] = c["date"].dt.isocalendar().week
    c["jour"] = c["date"].dt.day_of_week          # 0 = lundi
    return c


camions = pd.concat([preparer(avant, "Avant"), preparer(apres, "Après")], ignore_index=True)
connus = camions[~camions["reserve"]]   # camions dont le poids est connu, les seuls pour analyser le remplissage

#  Contrôle pour etre sur qu'on retrouve bien les chiffres des milestones 2 et 3 
controle = camions.groupby("version", sort=False).agg(camions=("camion_id", "size"), dont_reserves=("reserve", "sum"),
                                                      poids_total_t=("poids", "sum"), cout=("cout", "sum"))
controle["poids_total_t"] = (controle["poids_total_t"] / 1000).round(0)   # tonnes : l'écart d'arrondi (< 1 kg) disparaît
controle

Les 134 camions réservés sont exclus : on ne connaît pas leur poids. Il reste les camions dont le poids est connu, 1 607 avant (1 741 − 134) et 653 après (787 − 134). Comme leur nombre est très différent, on affiche la **part** des camions de chaque version dans chaque classe, et non leur nombre.

### Nouvelles distributions (taux de remplissage, coûts, ...)

#### Poids et taux de remplissage des camions

Le taux de remplissage d'un camion, c'est son poids divisé par 24,5 t. Les classes suivent les tranches de poids de la grille tarifaire, car le milestone 2 a montré que le seuil de 4 000 kg est décisif : au-delà, le prix d'un camion ne bouge presque plus. Sous chaque classe, on affiche le taux de remplissage correspondant.


In [ ]:
#  Poids et taux de remplissage par camion, selon les tranches de la grille tarifaire 
# Les bornes sont toutes des tranches de poids de la grille : dans une classe, le prix d'un camion change peu
BORNES_KG = [0, 100, 300, 500, 1000, 2000, 4000, POIDS_MAX]
# Chaque classe affiche aussi son taux de remplissage (poids / 24,5 t)
LIBELLES_KG = [f"{a}-{b} kg\n({pourcent(BORNES_KG[i] / POIDS_MAX)}-{pourcent(BORNES_KG[i + 1] / POIDS_MAX)})"
               for i, (a, b) in enumerate(zip(["0", "100", "300", "500", "1 000", "2 000", "4 000"],
                                              ["100", "300", "500", "1 000", "2 000", "4 000", "24 500"]))]

effectifs = connus.groupby("version")["camion_id"].size()
classes_kg = (pd.crosstab(pd.cut(connus["poids"], bins=BORNES_KG, labels=LIBELLES_KG, include_lowest=True),
                          connus["version"], normalize="columns")     # part des camions de chaque version
              [["Avant", "Après"]])
mediane_kg = connus.groupby("version")["poids"].median()

ax = classes_kg.mul(100).plot(kind="bar", color=[COULEURS[v] for v in classes_kg.columns], edgecolor="black",
                              figsize=(11, 5.2), rot=0, width=0.8)
for conteneur in ax.containers:
    ax.bar_label(conteneur, fmt="%.0f %%", fontsize=8, padding=2)
ax.axvline(len(LIBELLES_KG) - 1.5, color="red", linestyle="--")      # entre « 2 000-4 000 » et « > 4 000 »
ax.annotate("fin de la grille tarifaire :\nau-delà, prix quasi fixe", xy=(len(LIBELLES_KG) - 1.45, 22),
            color="red", fontsize=9)
ax.set_title(f"Poids et taux de remplissage des camions, selon les tranches de la grille tarifaire\n"
             f"médiane : {nombre(mediane_kg['Avant'], 0)} kg ({pourcent(mediane_kg['Avant'] / POIDS_MAX)}) avant → "
             f"{nombre(mediane_kg['Après'], 0)} kg ({pourcent(mediane_kg['Après'] / POIDS_MAX)}) après "
             f"(hors camions réservés)")
ax.set_xlabel("Poids du camion (taux de remplissage = poids / 24,5 t)")
ax.set_ylabel("Part des camions (%)")
ax.legend(ax.containers, [f"Avant : baseline ({effectifs['Avant']} camions)",
                          f"Après : planning consolidé ({effectifs['Après']} camions)"], loc="upper center")
ax.grid(axis="y", alpha=0.5)
ax.set_axisbelow(True)
ax.margins(y=0.12)
plt.tight_layout()
plt.show()

# ---------- Chiffres pour l'analyse ----------
for version in ["Avant", "Après"]:
    p = connus.loc[connus["version"] == version, "poids"]
    print(f"{version} : médiane {nombre(p.median(), 0)} kg ({pourcent(p.median() / POIDS_MAX)}), "
          f"moyenne {nombre(p.mean(), 0)} kg | ≤ 500 kg : {pourcent((p <= 500).mean())} | "
          f"> 4 000 kg : {(p > 4000).sum()} camions ({pourcent((p > 4000).mean())}), "
          f"soit {pourcent(p[p > 4000].sum() / p.sum())} du poids transporté")

# Pourquoi le remplissage reste bas : le poids qu'un couple a à livrer en une semaine
poids_semaine = connus[connus["version"] == "Après"].groupby(["supplier_id", "plant", "semaine"])["poids"].sum()
print(f"Poids médian à livrer par couple et par semaine : {nombre(poids_semaine.median(), 0)} kg ; "
      f"{pourcent((poids_semaine <= 4000).mean())} des couples-semaines sous 4 000 kg")

#### Analyse : poids et taux de remplissage

**La distribution se décale vers la droite.** Le poids médian d'un camion passe de **323 kg à 687 kg**, soit un taux de remplissage médian de **1,3 % à 2,8 %** (×2,1). Les petits envois reculent nettement : 64 % des camions pesaient moins de 500 kg (2 % de remplissage) dans la baseline, contre 42 % après consolidation. La classe 100-300 kg, la plus fréquente avant (30 % des camions), tombe à 19 %.

**Le même poids part dans 2,5 fois moins de camions.** Les 1 588 t à livrer sont les mêmes avant et après (aucune ligne perdue, voir la partie 1.4). Elles partent dans 653 camions au lieu de 1 607. Le poids moyen d'un camion passe donc de 988 kg à 2 433 kg.

**La part des camions au-delà de 4 000 kg est multipliée par 3,5 : de 4,4 % à 15,2 %** (de 71 à 99 camions). C'est le chiffre qui compte pour le prix : au-delà de 4 000 kg, le prix d'un camion est quasi fixe. Ces camions transportent maintenant **69 % du poids total**, contre 47 % avant : la majorité des marchandises voyage dans la tranche de prix la plus avantageuse au kilo.

**Le remplissage reste pourtant faible**, car les couples ont peu à livrer : en médiane, un couple n'a que 687 kg à livrer par semaine, et 86 % des couples-semaines restent sous 4 000 kg. Même regroupés dans un seul camion par semaine, ces besoins ne peuvent pas remplir un camion de 24,5 t. Pour aller plus loin, il faudrait lever deux contraintes : mélanger plusieurs fournisseurs dans un même camion (C4) ou livrer plus de 7 jours en avance (C6).

#### Coût de transport par tranche de poids

Le graphe précédent compte des camions ; celui-ci additionne leur coût, avec les mêmes tranches de poids. Sous chaque tranche, on affiche le coût moyen à la tonne transportée : c'est lui qui montre l'effet de la grille tarifaire dégressive. Les camions réservés sont exclus (leur coût est identique avant et après : 33 506 €).

In [ ]:
#  Où part l'argent : coût total par tranche de poids, avant / après 
classe_cout = pd.cut(connus["poids"], bins=BORNES_KG, labels=LIBELLES_KG, include_lowest=True)
cout_classe = connus.groupby([classe_cout, "version"], observed=False)["cout"].sum().unstack()[["Avant", "Après"]]
cout_tonne = connus.groupby("version").apply(lambda c: c["cout"].sum() / c["poids"].sum() * 1000)   # € par tonne
cout_tonne_classe = connus.groupby(classe_cout, observed=False).apply(lambda c: c["cout"].sum() / c["poids"].sum() * 1000)

ax = cout_classe.plot(kind="bar", color=[COULEURS[v] for v in cout_classe.columns], edgecolor="black",
                      figsize=(11, 5.2), rot=0, width=0.8)
for conteneur in ax.containers:
    ax.bar_label(conteneur, labels=[eur(v) for v in conteneur.datavalues], fontsize=8, padding=2)
ax.axvline(len(LIBELLES_KG) - 1.5, color="red", linestyle="--")
ax.set_xticks(range(len(LIBELLES_KG)),
              [f"{l.split(chr(10))[0]}\n{nombre(t, 0)} €/t" for l, t in zip(LIBELLES_KG, cout_tonne_classe)])
total = connus.groupby("version")["cout"].sum()
ax.set_title(f"Coût de transport par tranche de poids des camions (hors camions réservés)\n"
             f"total : {eur(total['Avant'])} avant → {eur(total['Après'])} après ; "
             f"coût à la tonne : {nombre(cout_tonne['Avant'], 0)} €/t → {nombre(cout_tonne['Après'], 0)} €/t")
ax.set_xlabel("Poids du camion (sous chaque tranche : coût moyen à la tonne transportée)")
ax.set_ylabel("Coût total sur la période (€)")
ax.legend(ax.containers, ["Avant : baseline", "Après : planning consolidé"], loc="upper center", ncol=2)
ax.grid(axis="y", alpha=0.5)
ax.set_axisbelow(True)
ax.set_ylim(0, cout_classe.values.max() * 1.25)
ax.yaxis.set_major_formatter(lambda v, _: nombre(v, 0))
plt.tight_layout()
plt.show()

# Chiffres pour l'analyse
for version in ["Avant", "Après"]:
    c = connus[connus["version"] == version]
    petits = c["poids"] <= 500
    print(f"{version} : coût total {eur(c['cout'].sum())}, coût médian d'un camion {nombre(c['cout'].median(), 0)} €, "
          f"coût à la tonne {nombre(cout_tonne[version], 0)} €/t | camions ≤ 500 kg : {pourcent(petits.mean())} des camions, "
          f"{pourcent(c.loc[petits, 'poids'].sum() / c['poids'].sum())} du poids, "
          f"{pourcent(c.loc[petits, 'cout'].sum() / c['cout'].sum())} du coût ({eur(c.loc[petits, 'cout'].sum())})")

#### Analyse : coût par tranche de poids

**La grille tarifaire rend les petits camions très chers à la tonne.** Une tonne transportée dans un camion de moins de 100 kg coûte en moyenne 778 €, contre 23 € dans un camion de plus de 4 000 kg : environ 34 fois moins. Ce coût à la tonne dépend de la grille, pas du moteur. L'économie vient donc du fait que le poids est déplacé vers les tranches les moins chères.

**C'est ce que fait la consolidation.** Dans la baseline, les camions de moins de 500 kg représentaient 64 % des camions et seulement 13 % du poids transporté, mais **39 % du coût** (55 807 €). Après consolidation, ils ne pèsent plus que 20 % du coût (16 510 €). Toutes les tranches sous 4 000 kg baissent, surtout les plus légères.

**Seule la tranche au-delà de 4 000 kg coûte plus cher après** (17 182 € → 25 064 €). C'est voulu : elle compte plus de camions, mais c'est la tranche où la tonne coûte le moins cher. Elle transporte désormais 69 % du poids pour 30 % du coût.

**Au total, hors camions réservés, le coût passe de 143 815 € à 82 612 € (−43 %), et le coût à la tonne de 91 €/t à 52 €/t.** Chaque camion coûte un peu plus cher (médiane de 71 € à 101 €, car il est plus lourd), mais il y en a 2,5 fois moins.

#### Camions reçus par jour et par usine

Ce graphe reprend la charge des usines du milestone 2, jour par jour et usine par usine. Les camions réservés sont comptés : l'usine les reçoit à ses quais comme les autres. Le milestone 3 comparait le planning avant et après lissage. Ici, on le compare à la situation actuelle (baseline).

In [ ]:
#  Camions reçus chaque jour ouvré, par usine : avant / après 
# Tous les camions sont comptés (camions réservés compris) 
par_jour = (camions.groupby(["plant", "date", "version"]).size().unstack(fill_value=0)
                   .reindex(columns=["Avant", "Après"]))
dates = sorted(camions["date"].unique())
lundis = [d for d in dates if pd.Timestamp(d).day_of_week == 0]

fig, axes = plt.subplots(2, 2, figsize=(14, 7.5), sharex=True)
for ax, plant in zip(axes.flat, sorted(camions["plant"].unique())):
    serie = par_jour.loc[plant].reindex(dates, fill_value=0)
    x = np.arange(len(dates))
    ax.bar(x, serie["Avant"], color=COULEURS["Avant"], edgecolor="black", linewidth=0.4, label="Avant : baseline")
    ax.bar(x, serie["Après"], width=0.45, color=COULEURS["Après"], label="Après : planning consolidé")
    for i, d in enumerate(dates):                         # un trait au début de chaque semaine
        if pd.Timestamp(d).day_of_week == 0:
            ax.axvline(i - 0.5, color="grey", linewidth=0.6, linestyle=":")
    ax.set_title(f"{plant} : jour le plus chargé {serie['Avant'].max()} → {serie['Après'].max()} camions")
    ax.set_xticks([dates.index(l) for l in lundis], [pd.Timestamp(l).strftime("%d/%m") for l in lundis])
    ax.grid(axis="y", alpha=0.5)
    ax.set_axisbelow(True)
for ax in axes[:, 0]:
    ax.set_ylabel("Camions reçus")
for ax in axes[1]:
    ax.set_xlabel("Jours ouvrés (étiquette = lundi de chaque semaine)")
axes[0, 0].legend(loc="upper left")
fig.suptitle("Camions reçus par jour ouvré et par usine (camions réservés compris)", fontsize=14)
plt.tight_layout()
plt.show()

#  Chiffres pour l'analyse 
stats = par_jour.reset_index().melt(id_vars=["plant", "date"], var_name="version", value_name="nb")
stats = stats[stats["nb"] > 0]   # moyenne sur les jours où l'usine reçoit des camions (pas sur les jours hors planning)
print(stats.groupby(["plant", "version"])["nb"].agg(["max", "mean", "std"]).round(1).unstack())
total_jour = camions.groupby(["date", "version"]).size().unstack()[["Avant", "Après"]]
print(f"Toutes usines : jour le plus chargé {int(total_jour['Avant'].max())} ({total_jour['Avant'].idxmax():%d/%m}) → "
      f"{int(total_jour['Après'].max())} ({total_jour['Après'].idxmax():%d/%m})")

#### Analyse : camions reçus par jour et par usine

**Toutes les usines reçoivent beaucoup moins de camions chaque jour.** En moyenne, sur les jours où elles sont livrées, PLT_001 passe de 18,6 à 7,0 camions par jour, PLT_002 de 19,7 à 9,1, PLT_003 de 6,9 à 3,7 et PLT_004 de 18,5 à 6,3.

**Les pics disparaissent.** Le cas le plus net est PLT_002 : ses pics des 26 et 27/11 (53 et 54 camions, repérés dès le milestone 1) tombent à 15 et 13 camions, et l'écart-type de sa charge quotidienne est divisé par 4 (de 14,3 à 3,6). Le jour le plus chargé de chaque usine baisse fortement : de 31 à 11 camions pour PLT_001, de 54 à 15 pour PLT_002, de 16 à 6 pour PLT_003 et de 24 à 10 pour PLT_004. Toutes usines confondues, le jour le plus chargé passe de 118 camions (le 26/11) à 35 (le 23/11).

**Certains jours reçoivent plus de camions qu'avant**, par exemple PLT_002 le 18/11 (13 camions au lieu de 3). C'est l'effet du lissage : des jours fixes sont placés sur les jours creux pour équilibrer la semaine.

**Des livraisons commencent avant le planning, du 03 au 06/11.** Les besoins du début du planning sont livrés en avance, sur les jours fixes de la semaine précédente (57 camions, au plus 23 le même jour toutes usines confondues). Le premier jour du planning n'est donc plus un pic : le 09/11, les usines reçoivent 30 camions au total, contre 57 dans la baseline.

#### Jours de livraison des couples les plus actifs

Même carte de chaleur qu'au milestone 2 : les 10 couples qui livrent le plus souvent dans la baseline, avec à droite les mêmes couples après consolidation. Une livraison correspond à un couple et un jour : plusieurs camions le même jour comptent pour une seule livraison. Chaque case donne le nombre de livraisons ce jour de la semaine sur toute la période (le planning final peut en compter une de plus, car il livre aussi la semaine qui précède le planning).
Ces couples étaient ceux au plus fort potentiel : cette comparaison montre si nous avons su les optimiser.

In [ ]:
#  Jours de livraison des 10 couples les plus actifs 
JOURS = ["Lundi", "Mardi", "Mercredi", "Jeudi", "Vendredi"]
camions["couple"] = camions["supplier_id"] + " × " + camions["plant"]

# Une livraison = un couple, un jour (plusieurs camions le même jour comptent pour une livraison)
livraisons = camions.drop_duplicates(["version", "couple", "date"])
tables = {v: pd.crosstab(livraisons.loc[livraisons["version"] == v, "couple"],
                         livraisons.loc[livraisons["version"] == v, "jour"]).reindex(columns=range(5), fill_value=0)
          for v in ["Avant", "Après"]}
top_10 = tables["Avant"].sum(axis=1).nlargest(10).index      # mêmes couples qu'au milestone 2
nb_semaines = besoins["need_date"].dt.isocalendar().week.nunique()   # 6 semaines de besoins dans le planning

fig, axes = plt.subplots(1, 2, figsize=(13, 6.5), sharey=True)
for ax, version in zip(axes, ["Avant", "Après"]):
    heat = tables[version].reindex(top_10, fill_value=0)
    im = ax.imshow(heat.values, cmap="YlGnBu", vmin=0, vmax=nb_semaines, aspect="auto")
    for i in range(heat.shape[0]):
        for j in range(heat.shape[1]):
            v = heat.values[i, j]
            ax.text(j, i, v if v else "", ha="center", va="center", fontsize=9,
                    color="white" if v > nb_semaines / 2 else "black")
    ax.set_xticks(range(5), JOURS)
    ax.set_yticks(range(len(top_10)), top_10)
    titre = "Avant : baseline" if version == "Avant" else "Après : planning consolidé"
    ax.set_title(f"{titre}\n{int(heat.values.sum())} livraisons, soit {nombre(heat.values.sum() / nb_semaines / 10)} par couple et par semaine")
fig.colorbar(im, ax=axes, label=f"Nombre de livraisons sur les {nb_semaines} semaines", shrink=0.8)
fig.suptitle("Jours de livraison des 10 couples les plus actifs de la baseline", fontsize=14)
plt.show()

#  Chiffres pour l'analyse
par_sem = livraisons.groupby(["version", "couple", "semaine"])["date"].nunique()   # jours de livraison par semaine
moy = par_sem.groupby(["version", "couple"]).mean()
for v in ["Avant", "Après"]:
    print(f"{v} : {moy[v].mean():.2f} jours de livraison par semaine en moyenne (couples livrés), "
          f"{(moy[v] > 2).sum()} couples à plus de 2 jours par semaine sur {len(moy[v])}")

# Les couples qui livrent encore plus de 2 jours par semaine après : camions optimisés / camions réservés
apres_c = camions[camions["version"] == "Après"]
for couple in moy["Après"][moy["Après"] > 2].index:
    nb = apres_c[apres_c["couple"] == couple].groupby("reserve").size()
    print(f"  {couple} : {nb.get(False, 0)} camions optimisés, {nb.get(True, 0)} camions réservés")

#### Analyse : jours de livraison des couples les plus actifs

**Avant, ces couples livraient tous les jours.** Les 10 couples les plus actifs de la baseline livrent les 5 jours de la semaine, presque chaque semaine : 291 livraisons sur les 6 semaines, soit 4,8 par couple et par semaine. 8 d'entre eux livrent PLT_004.

**Après, chaque couple livre une seule fois par semaine, toujours le même jour.** Ces mêmes couples ne font plus que 61 livraisons. Chaque ligne n'a plus qu'une case foncée, le jour fixe du couple : par exemple le lundi pour SUP_00085 → PLT_004, ou le vendredi pour SUP_00011 → PLT_004. SUP_00107 → PLT_001 compte 7 mercredis : il livre aussi le mercredi 04/11, la semaine qui précède le planning, pour les premiers besoins.

**Un seul jour fixe suffit.** Les 10 couples ont tous un profil d'un seul jour, comme 172 couples sur 173 : la fenêtre de 7 jours permet à un camion par semaine de couvrir tous les besoins.

**Sur l'ensemble des couples**, le nombre moyen de jours de livraison par semaine passe de 2,0 à 1,1. Seuls 4 couples sur 180 livrent encore plus de 2 jours par semaine, contre 52 avant. Ces 4 couples ont surtout des lignes sans poids (par exemple 24 camions réservés sur 30 pour SUP_00129 → PLT_003) : leurs camions réservés partent le jour du besoin, en dehors des jours fixes. Pour eux, seule une meilleure qualité des données permettrait de regrouper les livraisons.

--- 

### Étude des économies réalisées (réalisé vs potentiel)

#### Les 10 couples les plus chers

Même classement qu'au milestone 2 : les 10 couples fournisseur × usine qui coûtent le plus cher dans la baseline, camions réservés compris. Pour chacun, on compare le coût avant et après consolidation. La partie hachurée est le coût des camions réservés (lignes sans poids) : il est identique avant et après, car ces lignes ne peuvent pas être consolidées.

In [ ]:
#  Coût des 10 couples les plus chers de la baseline : avant / après 
from matplotlib.patches import Patch

# Coût total de chaque couple, camions réservés compris (comme au milestone 2), et la part des camions réservés
cout_couple = camions.pivot_table(index="couple", columns="version", values="cout", aggfunc="sum", fill_value=0)[["Avant", "Après"]]
cout_reserve = (camions[camions["reserve"]]
                .pivot_table(index="couple", columns="version", values="cout", aggfunc="sum", fill_value=0)
                .reindex(index=cout_couple.index, columns=["Avant", "Après"], fill_value=0))
top_cher = cout_couple.nlargest(10, "Avant").index[::-1]          # du moins cher au plus cher (le plus cher en haut)

fig, ax = plt.subplots(figsize=(11, 6.5))
y = np.arange(len(top_cher))
h = 0.4
for decalage, version in [(h / 2, "Avant"), (-h / 2, "Après")]:
    total = cout_couple.loc[top_cher, version]
    reserve = cout_reserve.loc[top_cher, version]
    ax.barh(y + decalage, total - reserve, height=h, color=COULEURS[version], edgecolor="black",
            label="Avant : baseline" if version == "Avant" else "Après : planning consolidé")
    ax.barh(y + decalage, reserve, left=total - reserve, height=h, color=COULEURS[version], edgecolor="black",
            hatch="///", alpha=0.6)
    for yi, v in zip(y + decalage, total):
        ax.text(v, yi, f" {eur(v)}", va="center", fontsize=8)
ax.set_yticks(y, top_cher)
ax.set_xlabel("Coût total sur les 6 semaines (€)")
ax.xaxis.set_major_formatter(lambda v, _: nombre(v, 0))
avant_top, apres_top = cout_couple.loc[top_cher, "Avant"].sum(), cout_couple.loc[top_cher, "Après"].sum()
ax.set_title(f"Coût des 10 couples les plus chers de la baseline (camions réservés compris)\n"
             f"{eur(avant_top)} → {eur(apres_top)} ({pourcent(apres_top / avant_top - 1)})")
poignees, _ = ax.get_legend_handles_labels()
ax.legend(poignees + [Patch(facecolor="white", edgecolor="black", hatch="///")],
          ["Avant : baseline", "Après : planning consolidé", "dont camions réservés (poids inconnu)"], loc="lower right")
ax.grid(axis="x", alpha=0.5)
ax.set_axisbelow(True)
ax.margins(x=0.12)
plt.tight_layout()
plt.show()

#  Chiffres pour l'analyse 
detail_top = pd.DataFrame({"avant": cout_couple.loc[top_cher, "Avant"], "après": cout_couple.loc[top_cher, "Après"],
                           "dont réservés": cout_reserve.loc[top_cher, "Avant"]})
detail_top["économie"] = detail_top["avant"] - detail_top["après"]
detail_top["économie (%)"] = (detail_top["économie"] / detail_top["avant"] * 100).round(1)
print(detail_top[::-1].round(0))
reserve_top = detail_top["dont réservés"].sum()
print(f"Hors camions réservés : {eur(avant_top - reserve_top)} → {eur(apres_top - reserve_top)} "
      f"({pourcent((apres_top - reserve_top) / (avant_top - reserve_top) - 1)})")
print(f"Part des 10 couples dans le coût de la baseline : {pourcent(avant_top / cout_couple['Avant'].sum())} ; "
      f"part dans l'économie totale : {pourcent((avant_top - apres_top) / (cout_couple['Avant'].sum() - cout_couple['Après'].sum()))}")

#### Analyse : les 10 couples les plus chers

**Les couples aux données complètes baissent fortement.** SUP_00085 → PLT_004, le couple le plus cher aux données complètes, passe de 7 429 € à 1 988 € (−73 %). SUP_00146 → PLT_002 (−68 %), SUP_00144 → PLT_001 (−57 %), SUP_00059 → PLT_003 (−57 %) et SUP_00118 → PLT_001 (−54 %) suivent la même tendance. Ce sont tous des couples qui livraient 3 à 5 jours par semaine dans la baseline : les regrouper sur un jour fixe est ce qui rapporte le plus.

**Les couples aux poids manquants ne baissent presque pas.** Pour SUP_00153 → PLT_004 et SUP_00001 → PLT_004, l'économie est nulle : leur coût vient entièrement, ou presque, des camions réservés (lignes sans poids), qu'on ne peut pas consolider. SUP_00129 ne gagne que 15 % vers PLT_003 et 3 % vers PLT_001, pour la même raison. Nous l'avions dit au milestone 2 : pour ces couples, l'économie passe d'abord par la qualité des données.

**Au total, ces 10 couples passent de 48 749 € à 31 415 € (−35,6 %).** Hors camions réservés, la baisse atteint **−59,1 %** (de 29 343 € à 12 010 €), bien plus que la moyenne. Ces 10 couples pèsent 27,5 % du coût de la baseline et apportent 28,3 % de l'économie totale.

#### Zoom sur une semaine : l'exemple du cours

On reprend l'exemple du document du cours (semaine du 16/11, 5 couples très actifs), cette fois avec notre planning : pour chaque couple, le coût actuel (baseline) et le coût après notre optimisation. Les camions réservés sont exclus, car ils sont identiques avant et après.

In [ ]:
# coût de 5 couples sur la semaine du 16/11, avant / après 
SEMAINE = 47   # semaine du lundi 16/11
COUPLES_DIAPO = ["SUP_00085 × PLT_004", "SUP_00144 × PLT_001", "SUP_00059 × PLT_003",
                 "SUP_00080 × PLT_004", "SUP_00149 × PLT_004"]          # les 5 couples de l'exemple du cours

# Camions de la semaine, hors camions réservés (identiques avant et après)
sem = camions[~camions["reserve"] & (camions["semaine"] == SEMAINE) & camions["couple"].isin(COUPLES_DIAPO)]
diapo = sem.pivot_table(index="couple", columns="version", values="cout", aggfunc="sum").reindex(COUPLES_DIAPO)
nb_camions = sem.pivot_table(index="couple", columns="version", values="camion_id", aggfunc="size").reindex(COUPLES_DIAPO)

total_av, total_ap = diapo["Avant"].sum(), diapo["Après"].sum()
c85 = diapo.loc["SUP_00085 × PLT_004"]

fig = plt.figure(figsize=(15, 6))
grille = fig.add_gridspec(3, 2, width_ratios=[1.9, 1], hspace=0.35, wspace=0.08)
ax = fig.add_subplot(grille[:, 0])

y = np.arange(len(COUPLES_DIAPO))[::-1]          # le premier couple en haut
h = 0.38
for decalage, version, nom in [(h / 2, "Avant", "Coût actuel (€ / semaine)"), (-h / 2, "Après", "Coût après consolidation")]:
    barres = ax.barh(y + decalage, diapo[version], height=h, color=COULEURS[version], label=nom)
    ax.bar_label(barres, labels=[nombre(v, 0) for v in diapo[version]], padding=4, fontsize=11)
ax.set_yticks(y, COUPLES_DIAPO, fontsize=11)
ax.set_title("Coût de transport, semaine du 16/11 (€)", fontsize=15, fontweight="bold")
ax.xaxis.set_major_formatter(lambda v, _: nombre(v, 0))
ax.grid(axis="x", alpha=0.4)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
ax.margins(x=0.1)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.2), ncol=2, frameon=False, fontsize=10)

#  Les 3 cartes chiffrées 
cartes = [
    (f"{pourcent(total_ap / total_av - 1, 0)}",
     f"coût des 5 couples : {eur(total_av)} → {eur(total_ap)} / semaine", "#e0a84f"),
    (eur(total_av - total_ap), "économisés sur une semaine pour 5 couples", "#e8eff1"),
    (eur(c85["Avant"] - c85["Après"]),
     f"SUP_00085 × PLT_004 : {int(nb_camions.loc['SUP_00085 × PLT_004', 'Avant'])} camions ({nombre(c85['Avant'], 0)} €)\n→ "
     f"{int(nb_camions.loc['SUP_00085 × PLT_004', 'Après'])} camion le lundi ({nombre(c85['Après'], 0)} €)", "#e8eff1"),
]
for i, (chiffre, texte, couleur) in enumerate(cartes):
    carte = fig.add_subplot(grille[i, 1])
    carte.set_facecolor(couleur)
    carte.set_xticks([]), carte.set_yticks([])
    for bord in carte.spines.values():
        bord.set_visible(False)
    carte.text(0.06, 0.62, chiffre, fontsize=26, fontweight="bold", color="#1f3b4d", transform=carte.transAxes)
    carte.text(0.06, 0.15, texte, fontsize=10.5, color="#1f3b4d", transform=carte.transAxes, wrap=True)

fig.text(0.66, 0.02, "Camions réservés exclus (identiques avant et après).", fontsize=9, color="dimgray")
plt.show()

#  Chiffres pour l'analyse 
diapo["camions avant"] = nb_camions["Avant"]
diapo["camions après"] = nb_camions["Après"]
print(diapo.round(2).to_string())
print(f"Total : {eur(total_av)} → {eur(total_ap)} ({pourcent(total_ap / total_av - 1)}), économie {eur(total_av - total_ap)}")
print("Jour du camion après :", sem[sem["version"] == "Après"].groupby("couple")["date"].first().dt.strftime("%d/%m").to_dict())

#### Analyse : la semaine du 16/11

**Chaque couple passe de 5 camions à 1 seul dans la semaine.** Ces 5 couples livraient tous les jours dans la baseline. Avec notre planning, chacun livre une seule fois, le jour fixe de son profil. Le coût des 5 couples passe de **3 994 € à 1 263 € pour la semaine (−68 %)**, soit **2 732 € économisés**. Pour SUP_00085 → PLT_004, les 5 camions (1 215 €) sont remplacés par un seul le lundi 16/11 (331 €) : 884 € d'économie sur une seule semaine.

**On retrouve exactement les chiffres de l'exemple du cours, bien que notre moteur n'ait pas fait les mêmes choix.** Pour 4 couples sur 5, il ne livre pas le même jour que le cours, et le camion ne transporte donc pas exactement le même poids. Mais tous ces camions dépassent 4 000 kg : dans cette dernière tranche de la grille, le prix ne dépend plus que de la distance. Un seul camion par semaine coûte donc le même prix, quel que soit son jour. Comme dans le cours, les camions réservés sont exclus : ils sont identiques avant et après.

**Ces 5 couples sont les cas les plus favorables** : ils livrent beaucoup, tous les jours, et ont des données complètes. Leur économie (−68 %) est donc bien supérieure à celle de l'ensemble du planning (−34,5 %).

#### Baseline, estimation et planning final, par semaine et par usine

Au milestone 2, on avait estimé le gain possible avec un algorithme de base : chaque camion part le jour du plus ancien besoin et regroupe les 7 jours suivants, sans jours fixes ni lissage. C'est le **potentiel** de la consolidation. On recalcule ici cette estimation avec le même algorithme, puis on compare les trois situations : la baseline (avant), l'estimation (le potentiel) et le planning final du moteur (le réalisé). Les camions réservés sont comptés dans les trois : ils sont identiques.

In [ ]:
# Recalcul de l'estimation du milstone 2 
FENETRE = pd.Timedelta(days=7)   # avance maximale d'une livraison sur son besoin (SF-06)


def consolider_couple_estimation(lignes):
    """Camions d'un couple. Les lignes sont triées par date de besoin, puis du plus lourd au plus léger.
    Un camion part le jour du plus ancien besoin restant et prend les besoins des 7 jours suivants,
    tant qu'il reste de la place."""
    restantes = list(lignes.itertuples())
    camions_couple = []
    while restantes:
        date = restantes[0].need_date
        poids, volume, contenu, i = 0.0, 0.0, [], 0
        while i < len(restantes):
            ligne = restantes[i]
            if ligne.need_date > date + FENETRE:
                break                                  # hors fenêtre : les suivantes aussi, camion fermé
            if poids + ligne.weight > POIDS_MAX or volume + ligne.volume > VOLUME_MAX:
                i += 1                                 # ne rentre pas : on essaie la ligne suivante
                continue
            poids, volume = poids + ligne.weight, volume + ligne.volume
            contenu.append(ligne.Index)
            restantes.pop(i)
            i = 0                                      # on repart de la plus ancienne ligne restante
        camions_couple.append({"date": date, "poids": poids, "volume": volume, "lignes": contenu})
    return camions_couple


# Lignes au poids connu (silver + quarantaine) ; la ligne de plus de 24,5 t reste exclue, comme au milestone 2
a_consolider = (besoins[besoins["weight"].notna() & (besoins["weight"] <= POIDS_MAX)]
                .sort_values(["plant", "supplier_id", "need_date", "weight"], ascending=[True, True, True, False]))
estimation = pd.DataFrame([{"supplier_id": supplier, "plant": plant, **camion}
                           for (plant, supplier), lignes_couple in a_consolider.groupby(["plant", "supplier_id"])
                           for camion in consolider_couple_estimation(lignes_couple)])
estimation["reserve"] = False

# Contrôles (les mêmes qu'au milestone 2)
lignes_livrees = estimation["lignes"].explode()
assert (estimation["poids"] <= POIDS_MAX).all() and (estimation["volume"] <= VOLUME_MAX).all()   # capacité
assert lignes_livrees.is_unique and set(lignes_livrees) == set(a_consolider.index)              # chaque ligne une fois
assert abs(estimation["poids"].sum() - a_consolider["weight"].sum()) < 1e-6                     # aucun poids perdu

# Les lignes sans poids gardent leur camion réservé, le jour du besoin, comme dans la baseline
estimation = pd.concat([estimation.drop(columns="lignes"), avant[avant["reserve"]].drop(columns="camion_id")],
                       ignore_index=True)
estimation["camion_id"] = range(1, len(estimation) + 1)

# Les trois situations, avec les mêmes indicateurs (fonction preparer de la préparation)
NOMS = {"Avant": "Baseline", "Estimation": "Estimation (milestone 2)", "Après": "Planning final"}
trois = pd.concat([camions[camions["version"] == "Avant"], preparer(estimation, "Estimation"),
                   camions[camions["version"] == "Après"]], ignore_index=True)
trois["version"] = trois["version"].map(NOMS)
COULEURS_3 = {"Baseline": COULEURS["Avant"], "Estimation (milestone 2)": "#e0a84f", "Planning final": COULEURS["Après"]}

trois.groupby("version", sort=False).agg(camions=("camion_id", "size"), dont_reserves=("reserve", "sum"),
                                         cout=("cout", "sum")).style.format({"cout": eur})

In [ ]:
#  Camions et coût par semaine 
camions_sem = trois.pivot_table(index="semaine", columns="version", values="camion_id", aggfunc="size", fill_value=0)[list(NOMS.values())]
cout_sem = trois.pivot_table(index="semaine", columns="version", values="cout", aggfunc="sum", fill_value=0)[list(NOMS.values())]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, table, titre, fmt in [(axes[0], camions_sem, "Camions par semaine", lambda v: nombre(v, 0)),
                              (axes[1], cout_sem, "Coût de transport par semaine (€)", lambda v: nombre(v / 1000, 0) + " k")]:
    table.plot(kind="bar", ax=ax, color=[COULEURS_3[n] for n in table.columns], edgecolor="black", rot=0, width=0.85)
    for conteneur in ax.containers:
        ax.bar_label(conteneur, labels=[fmt(v) if v else "" for v in conteneur.datavalues], fontsize=7, padding=2)
    ax.set_xticks(range(len(table)), [f"S{s}" for s in table.index])
    ax.set_title(titre)
    ax.set_xlabel("Semaine")
    ax.yaxis.set_major_formatter(lambda v, _: nombre(v, 0))
    ax.margins(y=0.1)
    ax.grid(axis="y", alpha=0.4)
    ax.set_axisbelow(True)
    ax.legend(title=None)
fig.suptitle("Baseline, estimation du milestone 2 et planning final, par semaine (camions réservés compris)", fontsize=14)
plt.tight_layout()
plt.show()

#  Tableau par usine 
par_usine = trois.pivot_table(index="plant", columns="version", values="cout", aggfunc="sum")[list(NOMS.values())]
nb_usine = trois.pivot_table(index="plant", columns="version", values="camion_id", aggfunc="size")[list(NOMS.values())]
par_usine.loc["Total"] = par_usine.sum()
nb_usine.loc["Total"] = nb_usine.sum()
base, est, fin = (par_usine[n] for n in NOMS.values())
tableau_usines = pd.DataFrame({
    "Camions (baseline -> estimation -> final)": [f"{a} -> {b} -> {c}" for a, b, c in nb_usine.values],
    "Coût baseline": base, "Coût estimation": est, "Coût final": fin,
    "Gain estimé (potentiel)": (base - est) / base,
    "Gain réalisé": (base - fin) / base,
    "Part du potentiel captée": (base - fin) / (base - est),
    "Part du gain réalisé": (base - fin) / (base - fin)["Total"],
})
tableau_usines.style.format({"Coût baseline": eur, "Coût estimation": eur, "Coût final": eur,
                             "Gain estimé (potentiel)": pourcent, "Gain réalisé": pourcent,
                             "Part du potentiel captée": pourcent, "Part du gain réalisé": pourcent})

#### Analyse : baseline, estimation et planning final

**Le planning final suit de près l'estimation du milestone 2.** Il compte 787 camions et coûte 116 118 €, contre 692 camions et 109 513 € pour l'estimation, et 1 741 camions et 177 321 € dans la baseline. Il réalise un gain de **61 203 € (−34,5 %)**, soit **90 % du gain estimé** (67 807 €, −38,2 %). Les 10 % restants (6 604 €) sont le prix des règles que l'estimation ignorait : des jours fixes pour chaque couple, le lissage de la charge des usines, et l'obligation de livrer chaque jour du profil au moins une fois.

**Semaine par semaine, le planning final reste proche de l'estimation.** Il envoie entre 104 et 164 camions par semaine (S46 à S50), contre 239 à 402 dans la baseline. Le pic de S48 disparaît : 402 camions et 39 899 € dans la baseline, 164 camions et 24 200 € dans le planning final, contre 134 camions et 21 800 € dans l'estimation.

**Certaines semaines coûtent moins cher que l'estimation** (S46 et S51). Ce n'est pas une erreur : l'estimation n'est un maximum que sur l'ensemble de la période, car les camions ne partent pas les mêmes jours. Les besoins du début de S46 sont livrés en avance dès S45 (57 camions du 03 au 06/11, une semaine qui n'existait pas dans la baseline), et ceux de S51 en partie en S50.

**Par usine, le moteur capte entre 87 % et 92 % du gain estimé.** PLT_004 reste la cible prioritaire annoncée au milestone 2 : c'est elle qui gagne le plus (−39,8 %, 38,1 % du gain total), et elle capte 92 % de son potentiel. PLT_002 en capte le moins (87 %, 237 camions au lieu des 197 de l'estimation) : c'est l'usine dont la charge était la plus irrégulière (pic de S48), donc celle où les jours fixes et le lissage imposent le plus de camions en plus. PLT_003 gagne toujours le moins en pourcentage (−27,0 %), à cause de ses camions réservés, que ni l'estimation ni le moteur ne peuvent consolider.

#### Réalisé vs potentiel : d'où vient l'écart

Le graphe précédent montre que le planning final reste au-dessus de l'estimation du milestone 2. Ce tableau décompose cet écart, étape par étape du moteur : on part du potentiel (consolidation sans contrainte de jours), puis on ajoute une à une les règles métier. Les coûts viennent du bilan du milestone 3 (`data/gold/bilan.csv`) et comprennent les camions réservés.

In [ ]:
#  Réalisé vs potentiel, étape par étape 
BILAN_PATH = "../data/gold/bilan.csv"   # bilan des versions (milestone 3)
bilan = pd.read_csv(BILAN_PATH).set_index("Version")

baseline = bilan.loc["Baseline", "Coût"]
potentiel = bilan.loc["Algorithme de base (sans jours fixes)", "Coût"]      # = estimation du milestone 2
jours_fixes = bilan.loc["1 ou 2 jours fixes", "Coût"]
lissage = bilan.loc["1 ou 2 jours fixes + lissage", "Coût"]
realise = bilan.loc["1 ou 2 jours fixes + lissage + report", "Coût"]       # planning final
cout_reserves = camions.loc[camions["reserve"] & (camions["version"] == "Avant"), "cout"].sum()   # identique partout

etapes = pd.DataFrame([
    ("Baseline (situation actuelle)", baseline, "—"),
    ("Potentiel : consolidation sans contrainte de jours", potentiel, eur(potentiel - baseline)),
    ("+ jours fixes (1 ou 2 par couple, chaque jour du profil utilisé)", jours_fixes, f"+{eur(jours_fixes - potentiel)}"),
    ("+ lissage de la charge des usines", lissage, f"+{eur(lissage - jours_fixes)}"),
    ("+ report des lignes (réalisé, planning final)", realise, eur(realise - lissage)),
], columns=["Étape", "Coût", "Écart avec l'étape précédente"])
etapes["Économie vs baseline"] = [eur(baseline - c) + f" ({pourcent((baseline - c) / baseline)})" if c != baseline else "—"
                                  for c in etapes["Coût"]]
etapes["Coût"] = etapes["Coût"].map(eur)
display(etapes.style.hide(axis="index"))

print(f"Part du potentiel captée : {eur(baseline - realise)} sur {eur(baseline - potentiel)} = "
      f"{pourcent((baseline - realise) / (baseline - potentiel))}")
print(f"Coût des contraintes métier (jours fixes + lissage + report) : {eur(realise - potentiel)}")
print(f"Camions réservés (poids inconnu), identiques dans toutes les versions : {eur(cout_reserves)}, "
      f"soit {pourcent(cout_reserves / baseline)} de la baseline et {pourcent(cout_reserves / realise)} du planning final")

#### Analyse : réalisé vs potentiel

**Le planning final capte 90 % du gain possible.** Sans aucune contrainte de jours, la consolidation ferait économiser 67 807 € (−38,2 %) : c'est l'estimation du milestone 2. Le planning final économise **61 203 € (−34,5 %)**. Les règles métier coûtent donc **6 604 € sur les 6 semaines**, soit 10 % du gain possible.

**Les jours fixes représentent presque tout ce coût (+5 573 €).** Dans l'estimation, chaque camion part le jour exact du premier besoin. Avec un jour fixe, il part le dernier jour fixe avant ce besoin : sa fenêtre de 7 jours couvre donc moins de besoins à venir, et il faut plus de camions (766 au lieu de 692). La règle qui oblige à livrer au moins une fois chaque jour du profil rend un deuxième jour fixe peu intéressant : il impose des livraisons supplémentaires. Le moteur donne donc un seul jour fixe à 172 couples sur 173.

**Le lissage coûte peu au regard de ce qu'il apporte (+1 246 €).** Il ajoute 21 camions, mais ramène les semaines hors tolérance de 20 sur 20 à seulement 2. Sans lui, les couples d'une même usine choisiraient souvent le même jour, et les quais seraient saturés certains jours et vides d'autres. Le report des lignes sur le jour fixe suivant fait regagner 215 €, sans ajouter de camion.

**Le principal potentiel restant n'est pas dans l'algorithme, mais dans les données.** Les 134 lignes sans poids coûtent 33 506 € dans toutes les versions, soit 28,9 % du planning final, et le moteur ne peut pas les consolider. C'est 5 fois plus que le coût de toutes les règles métier réunies : renseigner les poids manquants rapporterait bien plus que n'importe quelle amélioration du moteur.

---

## Conclusion

**La consolidation change profondément la façon de livrer.**
- **Moins de camions, plus lourds** : on passe de 1 741 à 787 camions (÷ 2,2). Le poids médian d'un camion passe de 323 kg à 687 kg, et la part des camions au-delà de 4 000 kg (la dernière tranche de la grille, où le prix est fixe) de 4,4 % à 15,2 %. Ces camions transportent désormais 69 % du poids.
- **Moins cher à la tonne** : le coût moyen d'une tonne transportée passe de 91 € à 52 €. Les camions de moins de 500 kg, qui coûtaient 39 % du budget pour 13 % du poids, n'en coûtent plus que 20 %.
- **Des livraisons régulières** : 172 couples sur 173 livrent un seul jour fixe par semaine. Les couples les plus actifs passent de 5 livraisons par semaine à une seule. En moyenne, un couple livre 1,1 jour par semaine au lieu de 2,0.
- **Des usines moins chargées** : les pics de réception disparaissent. Le jour le plus chargé de PLT_002 passe de 54 à 15 camions, et celui de toutes les usines réunies de 118 à 35.

**L'économie est importante et proche du maximum possible.**
- Le coût total passe de **177 321 € à 116 118 € sur les 6 semaines, soit 61 203 € économisés (−34,5 %)**.
- C'est **90 % du potentiel** estimé au milestone 2 (67 807 €, sans aucune contrainte de jours). Les règles métier coûtent 6 604 € : surtout les jours fixes (+5 573 €), puis le lissage (+1 246 €), qui ramène les semaines hors tolérance de 20 à 2 sur 20.
- L'exemple du cours est retrouvé sur nos données : sur la semaine du 16/11, les 5 couples de l'exemple passent de 3 994 € à 1 263 € (−68 %).
- Toutes les usines y contribuent. PLT_004 apporte la plus grosse part (23 345 €, 38 % de l'économie).

**Le principal gisement restant est la qualité des données.** Les 134 lignes sans poids partent chacune dans un camion réservé, qui ne peut pas être consolidé : elles coûtent 33 506 € dans toutes les versions, soit 29 % du planning final, cinq fois plus que toutes les règles métier réunies. Ce sont elles qui expliquent les couples et les usines qui économisent le moins (SUP_00129, SUP_00153, PLT_003). Hors camions réservés, chaque usine économise entre 38 % et 46 %. Renseigner ces poids est le levier le plus efficace pour aller plus loin, devant toute amélioration de l'algorithme.

**Limites.** Le remplissage reste faible dans l'absolu (2,8 % en médiane), car la plupart des couples ont moins de 4 t de besoins par semaine. Pour remplir réellement les camions, il faudrait sortir du périmètre du projet : mélanger plusieurs fournisseurs dans un même camion (tournées), ou livrer plus de 7 jours en avance. Enfin, la ligne de plus de 24,5 t (SUP_00074, 26,9 t) reste exclue en attendant la réponse du client.